In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

ZIP_PATH = "/content/drive/MyDrive/GC1/Public_Jira_Dataset/2025-06-23_ThePublicJiraDataset.zip"

if os.path.exists(ZIP_PATH):
    size_gb = os.path.getsize(ZIP_PATH) / (1024**3)
    print("Dataset found successfully.")
    print(f"ZIP size: {size_gb:.2f} GB")
    print("Path:", ZIP_PATH)
else:
    print("ERROR: Dataset not found.")

Dataset found successfully.
ZIP size: 5.41 GB
Path: /content/drive/MyDrive/GC1/Public_Jira_Dataset/2025-06-23_ThePublicJiraDataset.zip


In [ ]:
!df -h /content

Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   21G   88G  20% /


In [ ]:
import zipfile

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    for info in z.infolist():
        if "DataDump" in info.filename:
            print(
                f"{info.file_size / (1024**3):.2f} GB",
                "->",
                info.filename
            )

0.00 GB -> ThePublicJiraDataset/3. DataDump/
0.00 GB -> ThePublicJiraDataset/3. DataDump/README.md
5.46 GB -> ThePublicJiraDataset/3. DataDump/mongodump-JiraReposAnon.archive


In [ ]:
import zipfile
import os

WORK_DIR = "/content/jira_work"
os.makedirs(WORK_DIR, exist_ok=True)

ARCHIVE_IN_ZIP = "ThePublicJiraDataset/3. DataDump/mongodump-JiraReposAnon.archive"

print("Extracting MongoDB archive...")
print("This may take some time.")

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    z.extract(ARCHIVE_IN_ZIP, WORK_DIR)

MONGO_ARCHIVE = os.path.join(
    WORK_DIR,
    ARCHIVE_IN_ZIP
)

print("\nExtraction complete.")
print("Archive:", MONGO_ARCHIVE)
print(
    "Size:",
    round(os.path.getsize(MONGO_ARCHIVE) / (1024**3), 2),
    "GB"
)

Extracting MongoDB archive...
This may take some time.

Extraction complete.
Archive: /content/jira_work/ThePublicJiraDataset/3. DataDump/mongodump-JiraReposAnon.archive
Size: 5.46 GB


In [ ]:
if os.path.exists(MONGO_ARCHIVE):
    print("MongoDB archive exists.")
    print(
        "Size:",
        round(os.path.getsize(MONGO_ARCHIVE)/(1024**3), 2),
        "GB"
    )
else:
    print("ERROR: MongoDB archive not found.")

MongoDB archive exists.
Size: 5.46 GB


In [ ]:
!wget -q https://fastdl.mongodb.org/tools/db/mongodb-database-tools-ubuntu2204-x86_64-100.13.0.tgz -O /content/mongodb-tools.tgz

!tar -xzf /content/mongodb-tools.tgz -C /content/

!sudo cp /content/mongodb-database-tools-ubuntu2204-x86_64-100.13.0/bin/* /usr/local/bin/

In [ ]:
!mongorestore --version

mongorestore version: 100.13.0
git version: 23008ff975be028544710a5da6ae749dc7e90ab7
Go version: go1.23.11
   os: linux
   arch: amd64
   compiler: gc


In [ ]:
!cat /etc/os-release

PRETTY_NAME="Ubuntu 24.04.5 LTS"
NAME="Ubuntu"
VERSION_ID="24.04"
VERSION="24.04.5 LTS (Noble Numbat)"
VERSION_CODENAME=noble
ID=ubuntu
ID_LIKE=debian
HOME_URL="https://www.ubuntu.com/"
SUPPORT_URL="https://help.ubuntu.com/"
BUG_REPORT_URL="https://bugs.launchpad.net/ubuntu/"
PRIVACY_POLICY_URL="https://www.ubuntu.com/legal/terms-and-policies/privacy-policy"
UBUNTU_CODENAME=noble
LOGO=ubuntu-logo


In [ ]:
!sudo apt-get update -qq

!sudo apt-get install -y gnupg curl

!curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc | \
sudo gpg --dearmor -o /usr/share/keyrings/mongodb-server-8.0.gpg

!echo "deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-8.0.gpg ] https://repo.mongodb.org/apt/ubuntu noble/mongodb-org/8.0 multiverse" | \
sudo tee /etc/apt/sources.list.d/mongodb-org-8.0.list

!sudo apt-get update -qq

!sudo apt-get install -y mongodb-org

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
gnupg is already the newest version (2.4.4-2ubuntu17.6).
gnupg set to manually installed.
curl is already the newest version (8.5.0-2ubuntu10.15).
0 upgraded, 0 newly installed, 0 to remove and 7 not upgraded.
deb [ arch=amd64,arm64 signed-by=/usr/share/keyrings/mongodb-server-8.0.gpg ] https://repo.mongodb.org/apt/ubuntu noble/mongodb-org/8.0 multiverse
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  mongodb-database

In [ ]:
!mkdir -p /content/mongodb_data

!mongod \
  --dbpath /content/mongodb_data \
  --bind_ip 127.0.0.1 \
  --port 27017 \
  --fork \
  --logpath /content/mongodb.log

about to fork child process, waiting until server is ready for connections.
forked process: 8176
child process started successfully, parent exiting


In [ ]:
!mongosh --quiet --eval "db.runCommand({ ping: 1 })"

]0;mongosh mongodb://127.0.0.1:27017/?directConnection=true&serverSelectionTimeoutMS=2000{ ok: 1 }
]0;

In [ ]:
!mongorestore \
  --archive="$MONGO_ARCHIVE" \
  --dryRun \
  --verbose \
  2>&1 | head -100

2026-10-06T08:35:50.417+0000	using write concern: &{majority <nil> 0s}
2026-10-06T08:35:50.425+0000	Failed: stream or file does not appear to be a mongodump archive
2026-10-06T08:35:50.425+0000	0 document(s) restored successfully. 0 document(s) failed to restore.


In [ ]:
import os

print("MONGO_ARCHIVE =", MONGO_ARCHIVE)
print("Exists:", os.path.exists(MONGO_ARCHIVE))
print("Is file:", os.path.isfile(MONGO_ARCHIVE))

if os.path.exists(MONGO_ARCHIVE):
    print("Size:", round(os.path.getsize(MONGO_ARCHIVE)/(1024**3), 2), "GB")

MONGO_ARCHIVE = /content/jira_work/ThePublicJiraDataset/3. DataDump/mongodump-JiraReposAnon.archive
Exists: True
Is file: True
Size: 5.46 GB


In [ ]:
!file "$MONGO_ARCHIVE"

/content/jira_work/ThePublicJiraDataset/3. DataDump/mongodump-JiraReposAnon.archive: gzip compressed data, original size modulo 2^32 352830924 gzip compressed data, unknown method, has CRC, extra field, encrypted, from FAT filesystem (MS-DOS, OS/2, NT), original size modulo 2^32 352830924


In [ ]:
import zipfile

with zipfile.ZipFile(ZIP_PATH, "r") as z:
    text = z.read(
        "ThePublicJiraDataset/3. DataDump/README.md"
    ).decode("utf-8", errors="replace")

print(text)

Command used to export the data (this command takes about 15 minutes to complete).

`mongodump --db=JiraReposAnon --gzip --archive=mongodump-JiraReposAnon.archive`

Accompanying command to restore the data (this command takes about 15 minutes to complete). Expanded, this data is ~60GB inside MongoDB.

`mongorestore --gzip --archive=mongodump-JiraReposAnon.archive --nsFrom "JiraReposAnon.*" --nsTo "JiraReposAnon.*"`

Change the `--nsTo` command to contain the desired name for the JiraRepos database.



For more information see: https://docs.mongodb.com/manual/tutorial/backup-and-restore-tools/



In [ ]:
!mongorestore \
  --gzip \
  --archive="$MONGO_ARCHIVE" \
  --nsFrom="JiraReposAnon.*" \
  --nsTo="JiraReposAnon.*"

2026-10-06T08:38:57.488+0000	preparing collections to restore from
2026-10-06T08:38:57.553+0000	reading metadata for JiraReposAnon.Hyperledger from archive '/content/jira_work/ThePublicJiraDataset/3. DataDump/mongodump-JiraReposAnon.archive'
2026-10-06T08:38:57.554+0000	reading metadata for JiraReposAnon.Mindville from archive '/content/jira_work/ThePublicJiraDataset/3. DataDump/mongodump-JiraReposAnon.archive'
2026-10-06T08:38:57.554+0000	reading metadata for JiraReposAnon.Sonatype from archive '/content/jira_work/ThePublicJiraDataset/3. DataDump/mongodump-JiraReposAnon.archive'
2026-10-06T08:38:57.554+0000	reading metadata for JiraReposAnon.Spring from archive '/content/jira_work/ThePublicJiraDataset/3. DataDump/mongodump-JiraReposAnon.archive'
2026-10-06T08:38:57.554+0000	reading metadata for JiraReposAnon.Qt from archive '/content/jira_work/ThePublicJiraDataset/3. DataDump/mongodump-JiraReposAnon.archive'
2026-10-06T08:38:57.554+0000	reading metadata for JiraReposAnon.Mojang from a

In [ ]:
!pip install -q pymongo

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.1/331.1 kB 2.5 MB/s eta 0:00:00


In [ ]:
from pymongo import MongoClient

client = MongoClient("mongodb://localhost:27017/")

print("Available databases:")
print(client.list_database_names())

Available databases:
['JiraReposAnon', 'admin', 'config', 'local']


In [ ]:
db = client["JiraReposAnon"]

collections = sorted(db.list_collection_names())

print("Number of collections:", len(collections))
print("\nCollections:")

for c in collections:
    print(c)

Number of collections: 16

Collections:
Apache
Hyperledger
IntelDAOS
JFrog
Jira
JiraEcosystem
MariaDB
Mindville
Mojang
MongoDB
Qt
RedHat
Sakai
SecondLife
Sonatype
Spring


In [ ]:
print(f"{'COLLECTION':25} {'DOCUMENTS':>15}")
print("-" * 42)

for c in collections:
    count = db[c].estimated_document_count()
    print(f"{c:25} {count:>15,}")

COLLECTION                      DOCUMENTS
------------------------------------------
Apache                          1,014,926
Hyperledger                        28,146
IntelDAOS                           9,474
JFrog                              15,535
Jira                              274,545
JiraEcosystem                      41,866
MariaDB                            31,229
Mindville                           2,134
Mojang                            420,819
MongoDB                           137,172
Qt                                148,579
RedHat                            353,000
Sakai                              50,550
SecondLife                          1,867
Sonatype                           87,284
Spring                             69,156


In [ ]:
sample = db["Apache"].find_one()

print("TOP-LEVEL FIELDS")
print("=" * 50)

for key, value in sample.items():
    print(f"{key:35} {type(value).__name__}")

TOP-LEVEL FIELDS
_id                                 ObjectId
expand                              str
id                                  str
self                                str
key                                 str
fields                              dict
changelog                           dict


In [ ]:
print("\nNESTED DICTIONARY FIELDS")
print("=" * 50)

for key, value in sample.items():
    if isinstance(value, dict):
        print(f"\n{key}:")
        for subkey in value.keys():
            print("   -", subkey)


NESTED DICTIONARY FIELDS

fields:
   - fixVersions
   - resolution
   - customfield_12312322
   - customfield_12312323
   - customfield_12310420
   - customfield_12312320
   - customfield_12312321
   - customfield_12312328
   - customfield_12312329
   - customfield_12312326
   - customfield_12310300
   - customfield_12312327
   - customfield_12312324
   - customfield_12312720
   - customfield_12312325
   - lastViewed
   - priority
   - labels
   - customfield_12312333
   - customfield_12312334
   - customfield_12313422
   - customfield_12312331
   - customfield_12312332
   - aggregatetimeoriginalestimate
   - timeestimate
   - customfield_12312330
   - versions
   - customfield_12311120
   - customfield_12313826
   - issuelinks
   - customfield_12312339
   - customfield_12313825
   - assignee
   - customfield_12312337
   - customfield_12313823
   - customfield_12312338
   - customfield_12311920
   - customfield_12313822
   - customfield_12312335
   - customfield_12313821
   - customfi

In [ ]:
fields = sample.get("fields", {})

important_fields = [
    "project",
    "summary",
    "description",
    "issuetype",
    "status",
    "priority",
    "resolution",
    "created",
    "updated",
    "resolutiondate",
    "assignee",
    "reporter",
    "creator",
    "labels",
    "components",
    "issuelinks",
    "comment",
    "fixVersions",
    "versions"
]

for field in important_fields:
    print("\n" + "=" * 70)
    print("FIELD:", field)
    print("-" * 70)

    value = fields.get(field, "NOT FOUND")

    if isinstance(value, dict):
        print("Type: dictionary")
        print("Keys:", list(value.keys()))

        # Print only small useful values
        for k in ["id", "key", "name", "displayName"]:
            if k in value:
                print(f"{k}: {value[k]}")

    elif isinstance(value, list):
        print("Type: list")
        print("Number of items:", len(value))

        if len(value) > 0:
            print("First item:", value[0])

    else:
        print(value)


FIELD: project
----------------------------------------------------------------------
Type: dictionary
Keys: ['self', 'id', 'key', 'name', 'projectTypeKey', 'avatarUrls', 'projectCategory']
id: 12310801
key: ZOOKEEPER
name: ZooKeeper

FIELD: summary
----------------------------------------------------------------------
Typo in the log message of ServerCnxnFactory

FIELD: description
----------------------------------------------------------------------
Found a small typo at https://github.com/apache/zookeeper/blame/master/zookeeper-server/src/main/java/org/apache/zookeeper/server/ServerCnxnFactory.java#L306 while trying to understand the log message generated by Solr: maxCnxns is not configured,&#8203; using default value 0.

FIELD: issuetype
----------------------------------------------------------------------
Type: dictionary
Keys: ['self', 'id', 'description', 'iconUrl', 'name', 'subtask', 'avatarId']
id: 4
name: Improvement

FIELD: status
-----------------------------------------

In [ ]:
print("TOP-LEVEL FIELDS")
print("=" * 70)

for key, value in sample.items():
    print(f"{key:35} {type(value).__name__}")

TOP-LEVEL FIELDS
_id                                 ObjectId
expand                              str
id                                  str
self                                str
key                                 str
fields                              dict
changelog                           dict


In [ ]:
changelog = sample["changelog"]

print("CHANGELOG TYPE:", type(changelog).__name__)
print("CHANGELOG KEYS:", list(changelog.keys()))

for key, value in changelog.items():
    if isinstance(value, list):
        print(f"{key}: {len(value)} items")
    else:
        print(f"{key}: {value}")

CHANGELOG TYPE: dict
CHANGELOG KEYS: ['startAt', 'maxResults', 'total', 'histories']
startAt: 0
maxResults: 3
total: 3
histories: 3 items


In [ ]:
histories = changelog.get("histories", [])

print("Number of history events:", len(histories))

if histories:
    first_history = histories[0]

    print("\nFIRST HISTORY EVENT")
    print("=" * 60)

    print("Keys:", list(first_history.keys()))
    print("Created:", first_history.get("created"))

    author = first_history.get("author", {})
    print("Author:", author.get("displayName"))

    print("\nChanges:")

    for item in first_history.get("items", []):
        print(
            "Field:", item.get("field"),
            "| From:", item.get("fromString"),
            "| To:", item.get("toString")
        )

Number of history events: 3

FIRST HISTORY EVENT
Keys: ['id', 'author', 'created', 'items']
Created: 2022-01-04T11:01:41.276+0000
Author: <<|author_displayName|1b1c2e3c-1121-41ee-9d1f-fd12913d64f8|>>

Changes:
Field: WorklogId | From: 703250 | To: None
Field: timeestimate | From: None | To: 0
Field: timespent | From: None | To: 600


In [ ]:
linked_sample = db["Apache"].find_one({
    "fields.issuelinks.0": {"$exists": True}
})

if linked_sample:
    f = linked_sample["fields"]

    print("Project:", f.get("project", {}).get("name"))
    print("Project Key:", f.get("project", {}).get("key"))
    print("Issue Key:", linked_sample.get("key"))
    print("Summary:", f.get("summary"))

    links = f.get("issuelinks", [])

    print("\nNumber of issue links:", len(links))
    print("\nFIRST LINK")
    print("=" * 60)
    print(links[0])
else:
    print("No linked issue found.")

Project: ZooKeeper
Project Key: ZOOKEEPER
Issue Key: ZOOKEEPER-4434
Summary: Backport ZOOKEEPER-3142 for branch-3.5

Number of issue links: 1

FIRST LINK
{'id': '12629622', 'self': 'https://issues.apache.org/jira/rest/api/2/issueLink/12629622', 'type': {'id': '10030', 'name': 'Reference', 'inward': 'is related to', 'outward': 'relates to', 'self': 'https://issues.apache.org/jira/rest/api/2/issueLinkType/10030'}, 'outwardIssue': {'id': '13184190', 'key': 'ZOOKEEPER-3142', 'self': 'https://issues.apache.org/jira/rest/api/2/issue/13184190', 'fields': {'summary': 'Extend SnapshotFormatter to dump data in json format', 'status': {'self': 'https://issues.apache.org/jira/rest/api/2/status/5', 'description': 'A resolution has been taken, and it is awaiting verification by reporter. From here issues are either reopened, or are closed.', 'iconUrl': 'https://issues.apache.org/jira/images/icons/statuses/resolved.png', 'name': 'Resolved', 'id': '5', 'statusCategory': {'self': 'https://issues.apache

In [ ]:
comment_fields = set()

for doc in db["Apache"].find().limit(5000):

    fields = doc.get("fields", {})

    for key in fields.keys():
        if "comment" in key.lower():
            comment_fields.add(key)

print("Comment-related fields found:")
print(comment_fields)

Comment-related fields found:
{'comments'}


In [ ]:
comment_sample = db["Apache"].find_one({
    "fields.comments": {"$exists": True, "$ne": None}
})

if comment_sample:
    comments = comment_sample["fields"]["comments"]

    print("Comments type:", type(comments).__name__)

    if isinstance(comments, list):
        print("Number of comments:", len(comments))

        if comments:
            print("First comment fields:",
                  list(comments[0].keys())
                  if isinstance(comments[0], dict)
                  else type(comments[0]).__name__)

    elif isinstance(comments, dict):
        print("Comment dictionary keys:",
              list(comments.keys()))

else:
    print("No issue with comments found.")

Comments type: list
Number of comments: 0


In [ ]:
comment_sample = db["Apache"].find_one({
    "fields.comments.0": {"$exists": True}
})

if comment_sample:
    f = comment_sample["fields"]
    comments = f["comments"]

    print("Project:", f.get("project", {}).get("name"))
    print("Project Key:", f.get("project", {}).get("key"))
    print("Issue Key:", comment_sample.get("key"))
    print("Summary:", f.get("summary"))

    print("\nComments type:", type(comments).__name__)
    print("Number of comments:", len(comments))

    if comments:
        print("\nFirst comment fields:")
        print(list(comments[0].keys()))

        print("\nFirst comment:")
        print(comments[0])

else:
    print("No issue with non-empty comments found.")

Project: ZooKeeper
Project Key: ZOOKEEPER
Issue Key: ZOOKEEPER-4431
Summary: Log4j vulnerabilities in Apache zookeeper

Comments type: list
Number of comments: 4

First comment fields:
['self', 'id', 'author', 'body', 'updateAuthor', 'created', 'updated']

First comment:
{'self': 'https://issues.apache.org/jira/rest/api/2/issue/13419561/comment/17469059', 'id': '17469059', 'author': {'self': '<<|author_self|>>', 'name': '<<|author_name|d1364831-538f-4063-8170-eb6f5c8e57c4|>>', 'key': '<<|author_key|65e0e8a0-c5ce-4bd5-ae43-21d33451de63|>>', 'avatarUrls': '<<|author_avatarUrls|>>', 'displayName': '<<|author_displayName|987af430-9544-4d1a-9fe9-e9625fb9f98c|>>', 'active': True, 'timeZone': '<<|author_timeZone|>>'}, 'body': 'Hello, would appreciate if we could give this a priority due to log4j 1.x are out of support since August 2015 and there are many security patches has been rolled out since then. Looking forward to hear from the community and contributors. Thanks.', 'updateAuthor': {'se

In [ ]:
# NOW CREATE FOLDER IN DRIVE
# Selection process started

import os

PROFILE_DIR = "/content/drive/MyDrive/GC1/Project_Selection"
os.makedirs(PROFILE_DIR, exist_ok=True)

PROFILE_FILE = os.path.join(
    PROFILE_DIR,
    "all_project_profiles.csv"
)

print("Results will be saved to:")
print(PROFILE_FILE)

Results will be saved to:
/content/drive/MyDrive/GC1/Project_Selection/all_project_profiles.csv


In [ ]:
from collections import defaultdict, Counter
from datetime import datetime
import pandas as pd
import math

def parse_date(value):
    if not value:
        return None

    try:
        return datetime.strptime(
            value[:23],
            "%Y-%m-%dT%H:%M:%S.%f"
        )
    except:
        try:
            return datetime.strptime(
                value[:19],
                "%Y-%m-%dT%H:%M:%S"
            )
        except:
            return None


profiles = {}

for repo in collections:

    print(f"\nProcessing repository: {repo}")

    collection = db[repo]

    cursor = collection.find(
        {},
        {
            "key": 1,
            "fields.project": 1,
            "fields.issuetype": 1,
            "fields.status": 1,
            "fields.priority": 1,
            "fields.resolution": 1,
            "fields.created": 1,
            "fields.updated": 1,
            "fields.resolutiondate": 1,
            "fields.assignee": 1,
            "fields.description": 1,
            "fields.comments": 1,
            "fields.issuelinks": 1,
            "fields.labels": 1,
            "fields.components": 1,
            "fields.fixVersions": 1,
            "changelog.histories": 1
        },
        no_cursor_timeout=True
    ).batch_size(500)

    try:

        for doc in cursor:

            fields = doc.get("fields", {})

            project = fields.get("project")

            if not isinstance(project, dict):
                continue

            project_id = str(project.get("id", ""))
            project_key = project.get("key", "")
            project_name = project.get("name", "")

            if not project_key:
                continue

            uid = f"{repo}::{project_key}"

            if uid not in profiles:

                profiles[uid] = {

                    "repository": repo,
                    "project_id": project_id,
                    "project_key": project_key,
                    "project_name": project_name,

                    "issue_count": 0,

                    "bug_count": 0,
                    "improvement_count": 0,
                    "task_count": 0,

                    "resolved_count": 0,
                    "unresolved_count": 0,

                    "high_priority_count": 0,

                    "issues_with_description": 0,

                    "comment_count": 0,
                    "issues_with_comments": 0,

                    "issue_link_count": 0,
                    "issues_with_links": 0,
                    "blocking_link_count": 0,

                    "history_event_count": 0,
                    "field_change_count": 0,

                    "status_change_count": 0,
                    "priority_change_count": 0,
                    "assignee_change_count": 0,
                    "resolution_change_count": 0,
                    "version_change_count": 0,

                    "reopen_count": 0,

                    "issue_types": set(),
                    "statuses": set(),
                    "priorities": set(),
                    "assignees": set(),
                    "link_types": set(),

                    "earliest_created": None,
                    "latest_activity": None,

                    "resolution_days_sum": 0.0,
                    "resolved_with_duration": 0
                }

            p = profiles[uid]

            # ----------------------------------
            # ISSUE COUNT
            # ----------------------------------

            p["issue_count"] += 1

            # ----------------------------------
            # ISSUE TYPE
            # ----------------------------------

            issue_type = fields.get("issuetype") or {}

            issue_type_name = (
                issue_type.get("name", "")
                if isinstance(issue_type, dict)
                else ""
            )

            if issue_type_name:
                p["issue_types"].add(issue_type_name)

            lower_type = issue_type_name.lower()

            if "bug" in lower_type:
                p["bug_count"] += 1

            if "improvement" in lower_type:
                p["improvement_count"] += 1

            if "task" in lower_type:
                p["task_count"] += 1

            # ----------------------------------
            # STATUS
            # ----------------------------------

            status = fields.get("status") or {}

            status_name = (
                status.get("name", "")
                if isinstance(status, dict)
                else ""
            )

            if status_name:
                p["statuses"].add(status_name)

            # ----------------------------------
            # PRIORITY
            # ----------------------------------

            priority = fields.get("priority") or {}

            priority_name = (
                priority.get("name", "")
                if isinstance(priority, dict)
                else ""
            )

            if priority_name:
                p["priorities"].add(priority_name)

                if priority_name.lower() in [
                    "highest",
                    "critical",
                    "blocker",
                    "major",
                    "high"
                ]:
                    p["high_priority_count"] += 1

            # ----------------------------------
            # RESOLUTION
            # ----------------------------------

            resolution = fields.get("resolution")

            if resolution:
                p["resolved_count"] += 1
            else:
                p["unresolved_count"] += 1

            # ----------------------------------
            # DESCRIPTION
            # ----------------------------------

            description = fields.get("description")

            if description:
                p["issues_with_description"] += 1

            # ----------------------------------
            # ASSIGNEE
            # ----------------------------------

            assignee = fields.get("assignee")

            if isinstance(assignee, dict):

                assignee_id = (
                    assignee.get("key")
                    or assignee.get("name")
                    or assignee.get("displayName")
                )

                if assignee_id:
                    p["assignees"].add(str(assignee_id))

            # ----------------------------------
            # COMMENTS
            # ----------------------------------

            comments = fields.get("comments", [])

            if isinstance(comments, list):

                comment_count = len(comments)

                p["comment_count"] += comment_count

                if comment_count > 0:
                    p["issues_with_comments"] += 1

            # ----------------------------------
            # ISSUE LINKS
            # ----------------------------------

            links = fields.get("issuelinks", [])

            if isinstance(links, list):

                p["issue_link_count"] += len(links)

                if links:
                    p["issues_with_links"] += 1

                for link in links:

                    if not isinstance(link, dict):
                        continue

                    link_type = link.get("type", {})

                    if isinstance(link_type, dict):

                        name = str(
                            link_type.get("name", "")
                        )

                        inward = str(
                            link_type.get("inward", "")
                        )

                        outward = str(
                            link_type.get("outward", "")
                        )

                        if name:
                            p["link_types"].add(name)

                        combined = (
                            name + " " +
                            inward + " " +
                            outward
                        ).lower()

                        if (
                            "block" in combined
                            or "depend" in combined
                        ):
                            p["blocking_link_count"] += 1

            # ----------------------------------
            # DATES
            # ----------------------------------

            created = parse_date(
                fields.get("created")
            )

            updated = parse_date(
                fields.get("updated")
            )

            resolved_date = parse_date(
                fields.get("resolutiondate")
            )

            if created:

                if (
                    p["earliest_created"] is None
                    or created < p["earliest_created"]
                ):
                    p["earliest_created"] = created

            activity_date = updated or created

            if activity_date:

                if (
                    p["latest_activity"] is None
                    or activity_date > p["latest_activity"]
                ):
                    p["latest_activity"] = activity_date

            if created and resolved_date:

                days = (
                    resolved_date - created
                ).total_seconds() / 86400

                if days >= 0:
                    p["resolution_days_sum"] += days
                    p["resolved_with_duration"] += 1

            # ----------------------------------
            # CHANGELOG
            # ----------------------------------

            changelog = doc.get("changelog", {})

            histories = (
                changelog.get("histories", [])
                if isinstance(changelog, dict)
                else []
            )

            if isinstance(histories, list):

                p["history_event_count"] += len(histories)

                for history in histories:

                    if not isinstance(history, dict):
                        continue

                    items = history.get("items", [])

                    if not isinstance(items, list):
                        continue

                    p["field_change_count"] += len(items)

                    for item in items:

                        if not isinstance(item, dict):
                            continue

                        field = str(
                            item.get("field", "")
                        ).lower()

                        old_value = str(
                            item.get("fromString", "")
                            or ""
                        ).lower()

                        new_value = str(
                            item.get("toString", "")
                            or ""
                        ).lower()

                        if field == "status":

                            p["status_change_count"] += 1

                            # Reopen-like transitions
                            if (
                                new_value in [
                                    "reopened",
                                    "open"
                                ]
                                and old_value in [
                                    "resolved",
                                    "closed",
                                    "done"
                                ]
                            ):
                                p["reopen_count"] += 1

                        elif field == "priority":
                            p["priority_change_count"] += 1

                        elif field == "assignee":
                            p["assignee_change_count"] += 1

                        elif field == "resolution":
                            p["resolution_change_count"] += 1

                        if (
                            "version" in field
                            or "fix version" in field
                        ):
                            p["version_change_count"] += 1

    finally:
        cursor.close()

    print(
        f"Finished {repo}. "
        f"Projects discovered so far: {len(profiles)}"
    )


Processing repository: Apache
Finished Apache. Projects discovered so far: 646

Processing repository: Hyperledger
Finished Hyperledger. Projects discovered so far: 678

Processing repository: IntelDAOS
Finished IntelDAOS. Projects discovered so far: 680

Processing repository: JFrog
Finished JFrog. Projects discovered so far: 690

Processing repository: Jira
Finished Jira. Projects discovered so far: 720

Processing repository: JiraEcosystem
Finished JiraEcosystem. Projects discovered so far: 821

Processing repository: MariaDB
Finished MariaDB. Projects discovered so far: 832

Processing repository: Mindville
Finished Mindville. Projects discovered so far: 839

Processing repository: Mojang
Finished Mojang. Projects discovered so far: 847

Processing repository: MongoDB
Finished MongoDB. Projects discovered so far: 874

Processing repository: Qt
Finished Qt. Projects discovered so far: 895

Processing repository: RedHat
Finished RedHat. Projects discovered so far: 1136

Processing r

In [ ]:
rows = []

for uid, p in profiles.items():

    row = p.copy()

    # Convert sets into counts
    row["issue_type_diversity"] = len(
        row.pop("issue_types")
    )

    row["status_diversity"] = len(
        row.pop("statuses")
    )

    row["priority_diversity"] = len(
        row.pop("priorities")
    )

    row["assignee_diversity"] = len(
        row.pop("assignees")
    )

    row["link_type_diversity"] = len(
        row.pop("link_types")
    )

    # Project duration
    if (
        row["earliest_created"]
        and row["latest_activity"]
    ):

        row["duration_days"] = (
            row["latest_activity"]
            - row["earliest_created"]
        ).days

    else:
        row["duration_days"] = None

    # Average resolution time
    if row["resolved_with_duration"] > 0:

        row["avg_resolution_days"] = (
            row["resolution_days_sum"]
            / row["resolved_with_duration"]
        )

    else:
        row["avg_resolution_days"] = None

    # Useful ratios
    issues = max(row["issue_count"], 1)

    row["resolved_ratio"] = (
        row["resolved_count"] / issues
    )

    row["unresolved_ratio"] = (
        row["unresolved_count"] / issues
    )

    row["comment_coverage"] = (
        row["issues_with_comments"] / issues
    )

    row["link_coverage"] = (
        row["issues_with_links"] / issues
    )

    row["description_coverage"] = (
        row["issues_with_description"] / issues
    )

    row["changes_per_issue"] = (
        row["field_change_count"] / issues
    )

    row["links_per_issue"] = (
        row["issue_link_count"] / issues
    )

    rows.append(row)


profiles_df = pd.DataFrame(rows)

print("Individual Jira projects discovered:",
      len(profiles_df))

profiles_df.head()

Individual Jira projects discovered: 1276


,repository,project_id,project_key,project_name,issue_count,bug_count,improvement_count,task_count,resolved_count,unresolved_count,...,link_type_diversity,duration_days,avg_resolution_days,resolved_ratio,unresolved_ratio,comment_coverage,link_coverage,description_coverage,changes_per_issue,links_per_issue
0,Apache,12310801,ZOOKEEPER,ZooKeeper,4263,2240,1159,508,2805,1458,...,15,4960,212.050137,0.657987,0.342013,0.841192,0.280554,0.953788,19.647666,0.461881
1,Apache,12322520,ZIPKIN,Apache Zipkin,1,0,0,0,0,1,...,1,0,NaN,0.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000
2,Apache,12311131,ZETACOMP,Zeta Components,123,72,26,10,27,96,...,4,3790,74.283883,0.219512,0.780488,0.406504,0.048780,0.959350,1.723577,0.065041
3,Apache,12316221,ZEPPELIN,Zeppelin,5614,2874,1643,588,3392,2222,...,17,2485,49.981663,0.604204,0.395796,0.758461,0.164767,0.872462,10.026185,0.264339
4,Apache,12323620,YUNIKORN,Apache YuniKorn,994,249,218,457,801,193,...,17,699,39.360904,0.805835,0.194165,0.683099,0.295775,0.909457,12.399396,0.430584


In [ ]:
profiles_df.to_csv(
    PROFILE_FILE,
    index=False
)

print("Saved successfully:")
print(PROFILE_FILE)

Saved successfully:
/content/drive/MyDrive/GC1/Project_Selection/all_project_profiles.csv


In [ ]:
print("Total projects:", len(profiles_df))

print("\nTop 20 projects by issue count:")

display(
    profiles_df[
        [
            "repository",
            "project_name",
            "project_key",
            "issue_count",
            "field_change_count",
            "comment_count",
            "issue_link_count",
            "reopen_count",
            "resolved_ratio",
            "duration_days"
        ]
    ]
    .sort_values(
        "issue_count",
        ascending=False
    )
    .head(20)
)

Total projects: 1276

Top 20 projects by issue count:


,repository,project_name,project_key,issue_count,field_change_count,comment_count,issue_link_count,reopen_count,resolved_ratio,duration_days
845,Mojang,Minecraft: Java Edition,MC,213845,1977688,516801,247230,34196,0.961196,3361
841,Mojang,Minecraft (Bedrock codebase),MCPE,144778,1015600,298214,145293,29736,0.969187,3407
886,Qt,Qt,QTBUG,97172,1209106,298305,34710,5227,0.797761,6678
1191,Sonatype,Community Support - Open Source Project Reposi...,OSSRH,74960,425447,306968,1865,485,0.963087,4723
852,MongoDB,Core Server,SERVER,58928,1267009,158505,53989,3250,0.898928,4654
701,Jira,Jira Server and Data Center,JRASERVER,47225,1292162,145974,46711,1801,0.819015,7271
711,Jira,Confluence Server and Data Center,CONFSERVER,43910,1688673,130769,40844,2800,0.849237,6792
1152,Sakai,Sakai,SAK,43351,463059,162657,30644,6138,0.901617,6312
116,Apache,Spark,SPARK,37443,668638,124650,15837,1947,0.935662,734779
463,Apache,Apache Flex,FLEX,35390,196844,104864,502,199,0.891890,6168


In [ ]:
import numpy as np
import pandas as pd

df = profiles_df.copy()

# Avoid division by zero
issues = df["issue_count"].clip(lower=1)

# -----------------------------
# SIZE-NORMALIZED METRICS
# -----------------------------

df["changes_per_issue"] = (
    df["field_change_count"] / issues
)

df["status_changes_per_issue"] = (
    df["status_change_count"] / issues
)

df["priority_changes_per_issue"] = (
    df["priority_change_count"] / issues
)

df["comments_per_issue"] = (
    df["comment_count"] / issues
)

df["links_per_issue"] = (
    df["issue_link_count"] / issues
)

df["blocking_links_per_issue"] = (
    df["blocking_link_count"] / issues
)

df["reopens_per_issue"] = (
    df["reopen_count"] / issues
)

# -----------------------------
# CLEAN DURATION
# -----------------------------

# Keep raw duration for traceability
df["duration_days_raw"] = df["duration_days"]

# Projects > 30 years are treated as timestamp anomalies
df["duration_days_clean"] = df["duration_days"].where(
    (df["duration_days"] >= 0) &
    (df["duration_days"] <= 30 * 365)
)

print("Projects:", len(df))
print(
    "Invalid/anomalous durations:",
    df["duration_days_clean"].isna().sum()
)

Projects: 1276
Invalid/anomalous durations: 2


In [ ]:
selection_metrics = [
    "issue_count",
    "field_change_count",
    "changes_per_issue",
    "status_changes_per_issue",
    "priority_changes_per_issue",
    "comment_count",
    "comments_per_issue",
    "issue_link_count",
    "links_per_issue",
    "blocking_link_count",
    "blocking_links_per_issue",
    "reopen_count",
    "reopens_per_issue",
    "resolved_ratio",
    "duration_days_clean",
    "issue_type_diversity",
    "status_diversity",
    "assignee_diversity"
]

summary = df[selection_metrics].describe(
    percentiles=[0.25, 0.50, 0.75, 0.90, 0.95]
).T

display(summary)

,count,mean,std,min,25%,50%,75%,90%,95%,max
issue_count,1276.0,2105.236677,9106.727254,1.0,55.000000,296.500000,1293.000000,4068.500000,8531.500000,2.138450e+05
field_change_count,1276.0,24909.358150,111824.835974,0.0,266.250000,1952.000000,11057.500000,42708.000000,93210.250000,1.977688e+06
changes_per_issue,1276.0,7.844585,5.549727,0.0,4.335324,6.699521,10.115807,13.975096,17.560964,7.400000e+01
status_changes_per_issue,1276.0,1.547495,0.961022,0.0,0.952939,1.366785,1.977406,2.866315,3.313988,7.093443e+00
priority_changes_per_issue,1276.0,0.074345,0.138590,0.0,0.008850,0.034171,0.079806,0.163668,0.300646,1.428571e+00
comment_count,1276.0,6936.328370,28408.543419,0.0,60.750000,570.500000,3058.750000,12529.000000,26948.500000,5.168010e+05
comments_per_issue,1276.0,2.251972,1.654883,0.0,1.141626,1.983558,3.032630,4.161241,5.221776,1.271297e+01
issue_link_count,1276.0,1134.051724,8701.604616,0.0,5.000000,51.000000,323.250000,1553.000000,3684.500000,2.472300e+05
links_per_issue,1276.0,0.262674,0.319160,0.0,0.058824,0.191419,0.376840,0.590594,0.766095,6.049220e+00
blocking_link_count,1276.0,128.966301,565.780335,0.0,0.000000,9.000000,54.000000,252.000000,572.500000,1.291400e+04


In [ ]:
# --------------------------------------------
# BASIC ELIGIBILITY
# --------------------------------------------

eligible = df[
    (df["issue_count"] >= 1000) &
    (df["changes_per_issue"] >= 5) &
    (
        df["duration_days_clean"].isna() |
        (df["duration_days_clean"] >= 365)
    )
].copy()

print("Total projects:", len(df))
print("Eligible projects:", len(eligible))


# --------------------------------------------
# BEHAVIOURAL FLAGS
# --------------------------------------------

eligible["high_change"] = (
    (eligible["changes_per_issue"] >= 10) |
    (eligible["status_changes_per_issue"] >= 2)
)

eligible["dependency_heavy"] = (
    (eligible["links_per_issue"] >= 0.38) |
    (eligible["blocking_links_per_issue"] >= 0.076)
)

eligible["regression_failure_like"] = (
    (eligible["reopens_per_issue"] >= 0.095) &
    (
        (eligible["resolved_ratio"] < 0.816) |
        (eligible["blocking_links_per_issue"] >= 0.076)
    )
)

eligible["recovery_improvement"] = (
    (eligible["reopens_per_issue"] >= 0.027) &
    (eligible["resolved_ratio"] >= 0.915)
)

eligible["communication_rich"] = (
    eligible["comments_per_issue"] >= 3.03
)

eligible["structurally_diverse"] = (
    (eligible["issue_type_diversity"] >= 7) &
    (eligible["status_diversity"] >= 5)
)


# Count number of behavioural characteristics
behavior_cols = [
    "high_change",
    "dependency_heavy",
    "regression_failure_like",
    "recovery_improvement",
    "communication_rich",
    "structurally_diverse"
]

eligible["behavior_count"] = (
    eligible[behavior_cols]
    .sum(axis=1)
)


print("\nBehaviour counts:")
for col in behavior_cols:
    print(
        f"{col:28}",
        int(eligible[col].sum())
    )

Total projects: 1276
Eligible projects: 335

Behaviour counts:
high_change                  213
dependency_heavy             177
regression_failure_like      21
recovery_improvement         88
communication_rich           144
structurally_diverse         200


In [ ]:
view_cols = [
    "repository",
    "project_name",
    "project_key",
    "issue_count",
    "changes_per_issue",
    "links_per_issue",
    "blocking_links_per_issue",
    "reopens_per_issue",
    "resolved_ratio",
    "comments_per_issue",
    "issue_type_diversity",
    "status_diversity",
    "behavior_count",
    "high_change",
    "dependency_heavy",
    "regression_failure_like",
    "recovery_improvement",
    "communication_rich",
    "structurally_diverse"
]

top50 = (
    eligible[view_cols]
    .sort_values(
        ["behavior_count", "issue_count"],
        ascending=[False, False]
    )
    .head(50)
)

display(top50)

,repository,project_name,project_key,issue_count,changes_per_issue,links_per_issue,blocking_links_per_issue,reopens_per_issue,resolved_ratio,comments_per_issue,issue_type_diversity,status_diversity,behavior_count,high_change,dependency_heavy,regression_failure_like,recovery_improvement,communication_rich,structurally_diverse
1031,RedHat,Tools (JBoss Tools),JBIDE,27295,12.639128,0.585565,0.106100,0.185858,0.954204,4.206778,12,6,6,True,True,True,True,True,True
284,Apache,Maven,MNG,6046,12.753060,0.807476,0.093285,0.113629,0.924413,4.452365,8,5,6,True,True,True,True,True,True
1037,RedHat,Red Hat CodeReady Studio (devstudio),JBDS,4221,18.583511,0.893153,0.198294,0.200426,0.964226,5.122720,11,5,6,True,True,True,True,True,True
1152,Sakai,Sakai,SAK,43351,10.681622,0.706881,0.106549,0.141588,0.901617,3.752093,7,9,5,True,True,True,False,True,True
116,Apache,Spark,SPARK,37443,17.857490,0.422963,0.048607,0.051999,0.935662,3.329060,21,5,5,True,True,False,True,True,True
1209,Spring,Spring Framework,SPR,17413,10.116809,0.573939,0.040717,0.062597,0.915293,3.829782,8,6,5,True,True,False,True,True,True
1036,RedHat,JBoss Enterprise Application Platform,JBEAP,17102,26.946907,1.823296,0.203836,0.107297,0.939539,2.929833,30,9,5,True,True,True,True,False,True
228,Apache,Jackrabbit Oak,OAK,9612,9.698294,0.765814,0.148356,0.047025,0.916563,4.257283,11,5,5,True,True,False,True,True,True
1025,RedHat,JBoss Enterprise Application Platform 4 and 5,JBPAPP,5932,13.837492,0.989211,0.431052,0.260452,1.000000,4.034390,10,2,5,True,True,True,True,True,False
958,RedHat,JBoss BPMS Platform,RHBPMS,4026,21.302782,0.517387,0.174615,0.064829,0.999503,3.918033,7,5,5,True,True,False,True,True,True


In [ ]:
import pandas as pd
import numpy as np

# ============================================================
# 1. CREATE A STABLE / CONTROL CATEGORY
# ============================================================

eligible["stable_control"] = (
    (eligible["reopens_per_issue"] < 0.027) &
    (eligible["blocking_links_per_issue"] < 0.076) &
    (eligible["resolved_ratio"] >= 0.816) &
    (eligible["changes_per_issue"] < 10)
)

# ============================================================
# 2. DEFINE ALL BEHAVIOURAL CATEGORIES
# ============================================================

categories = [
    "high_change",
    "dependency_heavy",
    "regression_failure_like",
    "recovery_improvement",
    "communication_rich",
    "structurally_diverse",
    "stable_control"
]

# ============================================================
# 3. NORMALIZED QUALITY FEATURES
# ============================================================

quality_features = [
    "issue_count",
    "field_change_count",
    "comment_count",
    "issue_link_count",
    "issue_type_diversity",
    "status_diversity"
]

for col in quality_features:

    # log transform because counts are highly skewed
    values = np.log1p(eligible[col])

    min_v = values.min()
    max_v = values.max()

    if max_v > min_v:
        eligible[col + "_norm"] = (
            (values - min_v) /
            (max_v - min_v)
        )
    else:
        eligible[col + "_norm"] = 0


# Overall information richness
eligible["quality_score"] = (
    eligible["issue_count_norm"] +
    eligible["field_change_count_norm"] +
    eligible["comment_count_norm"] +
    eligible["issue_link_count_norm"] +
    eligible["issue_type_diversity_norm"] +
    eligible["status_diversity_norm"]
) / 6


# ============================================================
# 4. SELECT PROJECTS WITH CATEGORY + REPOSITORY DIVERSITY
# ============================================================

selected_indices = []
repository_counts = {}

# Desired minimum representation
category_targets = {
    "regression_failure_like": 5,
    "recovery_improvement": 5,
    "stable_control": 5,
    "dependency_heavy": 5,
    "high_change": 5,
    "communication_rich": 3,
    "structurally_diverse": 2
}

MAX_PER_REPOSITORY = 6


for category, target in category_targets.items():

    candidates = eligible[
        eligible[category] == True
    ].copy()

    # Projects satisfying several behaviours get a small bonus
    candidates["selection_score"] = (
        candidates["quality_score"] +
        0.05 * candidates["behavior_count"]
    )

    candidates = candidates.sort_values(
        "selection_score",
        ascending=False
    )

    added = 0

    for idx, row in candidates.iterrows():

        if idx in selected_indices:
            continue

        repo = row["repository"]

        if repository_counts.get(repo, 0) >= MAX_PER_REPOSITORY:
            continue

        selected_indices.append(idx)

        repository_counts[repo] = (
            repository_counts.get(repo, 0) + 1
        )

        added += 1

        if added >= target:
            break


# ============================================================
# 5. FILL REMAINING SLOTS UNTIL EXACTLY 30
# ============================================================

remaining = eligible[
    ~eligible.index.isin(selected_indices)
].copy()

remaining["selection_score"] = (
    remaining["quality_score"] +
    0.05 * remaining["behavior_count"]
)

remaining = remaining.sort_values(
    "selection_score",
    ascending=False
)

for idx, row in remaining.iterrows():

    if len(selected_indices) >= 30:
        break

    repo = row["repository"]

    if repository_counts.get(repo, 0) >= MAX_PER_REPOSITORY:
        continue

    selected_indices.append(idx)

    repository_counts[repo] = (
        repository_counts.get(repo, 0) + 1
    )


# ============================================================
# 6. CREATE FINAL DATAFRAME
# ============================================================

selected30 = eligible.loc[selected_indices].copy()

selected30 = selected30.head(30)

print("Final projects selected:", len(selected30))

Final projects selected: 30


In [ ]:
def get_patterns(row):

    patterns = []

    if row["high_change"]:
        patterns.append("High Change")

    if row["dependency_heavy"]:
        patterns.append("Dependency Heavy")

    if row["regression_failure_like"]:
        patterns.append("Regression/Failure-like")

    if row["recovery_improvement"]:
        patterns.append("Recovery/Improvement")

    if row["communication_rich"]:
        patterns.append("Communication Rich")

    if row["structurally_diverse"]:
        patterns.append("Structurally Diverse")

    if row["stable_control"]:
        patterns.append("Stable/Control")

    return ", ".join(patterns)


selected30["selection_patterns"] = selected30.apply(
    get_patterns,
    axis=1
)

In [ ]:
final_cols = [
    "repository",
    "project_name",
    "project_key",
    "issue_count",
    "changes_per_issue",
    "links_per_issue",
    "blocking_links_per_issue",
    "reopens_per_issue",
    "resolved_ratio",
    "comments_per_issue",
    "issue_type_diversity",
    "status_diversity",
    "selection_patterns"
]

final30_display = selected30[final_cols].copy()

display(final30_display)

,repository,project_name,project_key,issue_count,changes_per_issue,links_per_issue,blocking_links_per_issue,reopens_per_issue,resolved_ratio,comments_per_issue,issue_type_diversity,status_diversity,selection_patterns
1036,RedHat,JBoss Enterprise Application Platform,JBEAP,17102,26.946907,1.823296,0.203836,0.107297,0.939539,2.929833,30,9,"High Change, Dependency Heavy, Regression/Fail..."
1031,RedHat,Tools (JBoss Tools),JBIDE,27295,12.639128,0.585565,0.106100,0.185858,0.954204,4.206778,12,6,"High Change, Dependency Heavy, Regression/Fail..."
1152,Sakai,Sakai,SAK,43351,10.681622,0.706881,0.106549,0.141588,0.901617,3.752093,7,9,"High Change, Dependency Heavy, Regression/Fail..."
1037,RedHat,Red Hat CodeReady Studio (devstudio),JBDS,4221,18.583511,0.893153,0.198294,0.200426,0.964226,5.122720,11,5,"High Change, Dependency Heavy, Regression/Fail..."
284,Apache,Maven,MNG,6046,12.753060,0.807476,0.093285,0.113629,0.924413,4.452365,8,5,"High Change, Dependency Heavy, Regression/Fail..."
116,Apache,Spark,SPARK,37443,17.857490,0.422963,0.048607,0.051999,0.935662,3.329060,21,5,"High Change, Dependency Heavy, Recovery/Improv..."
1078,RedHat,Red Hat Fuse,ENTESB,13529,16.862222,0.409639,0.059724,0.125508,0.967034,2.837165,10,15,"High Change, Dependency Heavy, Recovery/Improv..."
1209,Spring,Spring Framework,SPR,17413,10.116809,0.573939,0.040717,0.062597,0.915293,3.829782,8,6,"High Change, Dependency Heavy, Recovery/Improv..."
632,Apache,Ambari,AMBARI,25384,10.252718,0.089505,0.014734,0.049638,0.918098,3.260124,11,6,"High Change, Recovery/Improvement, Communicati..."
845,Mojang,Minecraft: Java Edition,MC,213845,9.248231,1.156118,0.001609,0.159910,0.961196,2.416708,1,5,"Dependency Heavy, Recovery/Improvement"


In [ ]:
print("REPOSITORY DISTRIBUTION")
print("=" * 50)

print(
    selected30["repository"]
    .value_counts()
)

print("\n\nBEHAVIOURAL COVERAGE")
print("=" * 50)

for category in categories:

    print(
        f"{category:30}",
        int(selected30[category].sum())
    )

REPOSITORY DISTRIBUTION
repository
RedHat       6
Apache       6
MongoDB      6
Jira         4
Qt           2
Sonatype     2
Sakai        1
Spring       1
Mojang       1
IntelDAOS    1
Name: count, dtype: int64


BEHAVIOURAL COVERAGE
high_change                    26
dependency_heavy               23
regression_failure_like        5
recovery_improvement           10
communication_rich             19
structurally_diverse           20
stable_control                 5


In [ ]:
SELECTED_FILE = (
    "/content/drive/MyDrive/GC1/"
    "Project_Selection/selected_30_projects.csv"
)

selected30.to_csv(
    SELECTED_FILE,
    index=False
)

print("Saved:")
print(SELECTED_FILE)

Saved:
/content/drive/MyDrive/GC1/Project_Selection/selected_30_projects.csv


In [ ]:
import os

EXPORT_ROOT = "/content/drive/MyDrive/GC1/Selected_30_Jira_Projects"

os.makedirs(EXPORT_ROOT, exist_ok=True)

print("Projects will be exported to:")
print(EXPORT_ROOT)

Projects will be exported to:
/content/drive/MyDrive/GC1/Selected_30_Jira_Projects


In [ ]:
print("Selected projects:", len(selected30))

display(
    selected30[
        [
            "repository",
            "project_name",
            "project_key",
            "issue_count",
            "selection_patterns"
        ]
    ]
)

Selected projects: 30


,repository,project_name,project_key,issue_count,selection_patterns
1036,RedHat,JBoss Enterprise Application Platform,JBEAP,17102,"High Change, Dependency Heavy, Regression/Fail..."
1031,RedHat,Tools (JBoss Tools),JBIDE,27295,"High Change, Dependency Heavy, Regression/Fail..."
1152,Sakai,Sakai,SAK,43351,"High Change, Dependency Heavy, Regression/Fail..."
1037,RedHat,Red Hat CodeReady Studio (devstudio),JBDS,4221,"High Change, Dependency Heavy, Regression/Fail..."
284,Apache,Maven,MNG,6046,"High Change, Dependency Heavy, Regression/Fail..."
116,Apache,Spark,SPARK,37443,"High Change, Dependency Heavy, Recovery/Improv..."
1078,RedHat,Red Hat Fuse,ENTESB,13529,"High Change, Dependency Heavy, Recovery/Improv..."
1209,Spring,Spring Framework,SPR,17413,"High Change, Dependency Heavy, Recovery/Improv..."
632,Apache,Ambari,AMBARI,25384,"High Change, Recovery/Improvement, Communicati..."
845,Mojang,Minecraft: Java Edition,MC,213845,"Dependency Heavy, Recovery/Improvement"


In [ ]:
import os
import json
import pandas as pd
import numpy as np

EXPORT_ROOT = "/content/drive/MyDrive/GC1/Selected_30_Jira_Projects"

os.makedirs(EXPORT_ROOT, exist_ok=True)

print("Export folder:")
print(EXPORT_ROOT)

print("\nSelected projects:", len(selected30))

Export folder:
/content/drive/MyDrive/GC1/Selected_30_Jira_Projects

Selected projects: 30


In [ ]:
from datetime import datetime
from bson import ObjectId


def json_safe(obj):
    """Convert MongoDB/Python objects to JSON-safe values."""

    if isinstance(obj, ObjectId):
        return str(obj)

    if isinstance(obj, datetime):
        return obj.isoformat()

    if isinstance(obj, dict):
        return {
            str(k): json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(obj, list):
        return [
            json_safe(v)
            for v in obj
        ]

    return obj


def get_name(value):
    """Get readable/anonymized identity from Jira user object."""

    if not isinstance(value, dict):
        return None

    return (
        value.get("displayName")
        or value.get("name")
        or value.get("key")
        or value.get("id")
    )

In [ ]:
def extract_status_history(histories):

    status_history = []

    if not isinstance(histories, list):
        return status_history

    for history in histories:

        if not isinstance(history, dict):
            continue

        changed_at = history.get("created")
        changed_by = get_name(history.get("author"))

        for item in history.get("items", []):

            if not isinstance(item, dict):
                continue

            if str(item.get("field", "")).lower() == "status":

                status_history.append({
                    "changed_at": changed_at,
                    "changed_by": changed_by,
                    "from_status": item.get("fromString"),
                    "to_status": item.get("toString")
                })

    return status_history

In [ ]:
def extract_issue_links(links):

    blocks = []
    blocked_by = []
    depends_on = []
    other_links = []

    if not isinstance(links, list):
        return blocks, blocked_by, depends_on, other_links

    for link in links:

        if not isinstance(link, dict):
            continue

        link_type = link.get("type", {}) or {}

        relation_name = str(
            link_type.get("name", "")
        )

        inward_name = str(
            link_type.get("inward", "")
        )

        outward_name = str(
            link_type.get("outward", "")
        )

        # Current issue -> outward issue
        if isinstance(link.get("outwardIssue"), dict):

            target = link["outwardIssue"]

            record = {
                "issue_id": target.get("id"),
                "issue_key": target.get("key"),
                "relation": outward_name or relation_name
            }

            relation = (
                outward_name + " " + relation_name
            ).lower()

            if "block" in relation:
                blocks.append(record)

            elif "depend" in relation:
                depends_on.append(record)

            else:
                other_links.append(record)

        # inward issue -> current issue
        if isinstance(link.get("inwardIssue"), dict):

            source = link["inwardIssue"]

            record = {
                "issue_id": source.get("id"),
                "issue_key": source.get("key"),
                "relation": inward_name or relation_name
            }

            relation = (
                inward_name + " " + relation_name
            ).lower()

            if "block" in relation:
                blocked_by.append(record)

            elif "depend" in relation:
                depends_on.append(record)

            else:
                other_links.append(record)

    return blocks, blocked_by, depends_on, other_links

In [ ]:
def extract_block_times(status_history):

    blocked_at = None
    unblocked_at = None

    for event in status_history:

        old_status = str(
            event.get("from_status") or ""
        ).lower()

        new_status = str(
            event.get("to_status") or ""
        ).lower()

        # Entering blocked state
        if "block" in new_status and "block" not in old_status:

            if blocked_at is None:
                blocked_at = event.get("changed_at")

        # Leaving blocked state
        if "block" in old_status and "block" not in new_status:

            unblocked_at = event.get("changed_at")

    return blocked_at, unblocked_at

In [ ]:
def extract_comments(comments):

    output = []

    if not isinstance(comments, list):
        return output

    for comment in comments:

        if not isinstance(comment, dict):
            continue

        output.append({
            "comment_id": comment.get("id"),
            "author": get_name(
                comment.get("author")
            ),
            "body": comment.get("body"),
            "created_at": comment.get("created"),
            "updated_at": comment.get("updated")
        })

    return output

In [ ]:
def make_gc1_ticket(doc, repository):

    fields = doc.get("fields", {}) or {}

    project = fields.get("project", {}) or {}
    issue_type = fields.get("issuetype", {}) or {}
    status = fields.get("status", {}) or {}
    priority = fields.get("priority", {}) or {}
    resolution = fields.get("resolution", {}) or {}

    histories = (
        doc.get("changelog", {})
        .get("histories", [])
    )

    status_history = extract_status_history(
        histories
    )

    blocked_at, unblocked_at = (
        extract_block_times(status_history)
    )

    original_links = fields.get(
        "issuelinks",
        []
    )

    (
        blocks,
        blocked_by,
        depends_on,
        other_links
    ) = extract_issue_links(original_links)

    comments = extract_comments(
        fields.get("comments", [])
    )

    # -----------------------------------------
    # COMPONENTS
    # -----------------------------------------

    components = []

    for component in fields.get(
        "components", []
    ):

        if isinstance(component, dict):
            components.append({
                "id": component.get("id"),
                "name": component.get("name")
            })

    # -----------------------------------------
    # FIX VERSIONS
    # -----------------------------------------

    fix_versions = []

    for version in fields.get(
        "fixVersions", []
    ):

        if isinstance(version, dict):
            fix_versions.append({
                "id": version.get("id"),
                "name": version.get("name"),
                "released": version.get("released"),
                "release_date": version.get("releaseDate")
            })

    # -----------------------------------------
    # AFFECTED VERSIONS
    # -----------------------------------------

    affected_versions = []

    for version in fields.get(
        "versions", []
    ):

        if isinstance(version, dict):
            affected_versions.append({
                "id": version.get("id"),
                "name": version.get("name"),
                "released": version.get("released"),
                "release_date": version.get("releaseDate")
            })

    # -----------------------------------------
    # STANDARDIZED GC1 RECORD
    # -----------------------------------------

    ticket = {

        # SOURCE / PROVENANCE
        "source_type": "JIRA",
        "source_repository": repository,

        # PROJECT
        "project_id": project.get("id"),
        "project_key": project.get("key"),
        "project_name": project.get("name"),

        # ISSUE
        "issue_id": doc.get("id"),
        "issue_key": doc.get("key"),
        "issue_type": issue_type.get("name"),

        # MAIN UNSTRUCTURED CONTENT
        "summary": fields.get("summary"),
        "description": fields.get("description"),

        # REQUIREMENT / FEATURE
        # Do NOT fabricate these.
        # They can later be populated by LLM extraction/entity resolution
        # or explicit Jira fields if identified.
        "related_requirement": None,
        "epic_feature": None,

        # STATUS
        "status": status.get("name"),
        "status_history": status_history,

        # PRIORITY
        "priority": priority.get("name"),

        # PEOPLE / OWNERSHIP
        "assignee": get_name(
            fields.get("assignee")
        ),

        "reporter": get_name(
            fields.get("reporter")
        ),

        "creator": get_name(
            fields.get("creator")
        ),

        # Not consistently explicit in public Jira
        "team": None,

        # TEMPORAL FIELDS
        "created_at": fields.get("created"),
        "updated_at": fields.get("updated"),
        "due_at": fields.get("duedate"),
        "resolved_at": fields.get("resolutiondate"),

        # BLOCKING TIMES
        "blocked_at": blocked_at,
        "unblocked_at": unblocked_at,

        # RESOLUTION
        "resolution": resolution.get("name"),

        # RELATIONSHIPS
        "blocks": blocks,
        "blocked_by": blocked_by,
        "depends_on": depends_on,
        "other_issue_links": other_links,

        # TEXTUAL EVIDENCE
        "comments": comments,

        # CLASSIFICATION / ORGANIZATION
        "labels": fields.get("labels", []),
        "components": components,

        # RELEASE / VERSION CONTEXT
        "fix_versions": fix_versions,
        "affected_versions": affected_versions,

        # PRESERVE ORIGINAL LINKS
        "original_issue_links": json_safe(
            original_links
        ),

        # COMPLETE TEMPORAL HISTORY
        "full_changelog": json_safe(
            histories
        )
    }

    return json_safe(ticket)

In [ ]:
test_doc = db["Apache"].find_one({
    "fields.project.key": "ZOOKEEPER"
})

test_ticket = make_gc1_ticket(
    test_doc,
    "Apache"
)

print(
    json.dumps(
        test_ticket,
        indent=2,
        ensure_ascii=False
    )[:12000]
)

{
  "source_type": "JIRA",
  "source_repository": "Apache",
  "project_id": "12310801",
  "project_key": "ZOOKEEPER",
  "project_name": "ZooKeeper",
  "issue_id": "13420390",
  "issue_key": "ZOOKEEPER-4435",
  "issue_type": "Improvement",
  "summary": "Typo in the log message of ServerCnxnFactory",
  "description": "Found a small typo at https://github.com/apache/zookeeper/blame/master/zookeeper-server/src/main/java/org/apache/zookeeper/server/ServerCnxnFactory.java#L306 while trying to understand the log message generated by Solr: maxCnxns is not configured,&#8203; using default value 0.",
  "related_requirement": null,
  "epic_feature": null,
  "status": "Open",
  "status_history": [],
  "priority": "Trivial",
  "assignee": null,
  "reporter": "<<|author_displayName|53583068-c7f1-4a40-9e36-88645ea00e23|>>",
  "creator": "<<|author_displayName|53583068-c7f1-4a40-9e36-88645ea00e23|>>",
  "team": null,
  "created_at": "2022-01-03T14:50:04.000+0000",
  "updated_at": "2022-01-04T11:01:43.

In [ ]:
required_fields = [
    "project_id",
    "issue_id",
    "issue_type",
    "summary",
    "description",
    "related_requirement",
    "epic_feature",
    "status",
    "status_history",
    "priority",
    "assignee",
    "team",
    "created_at",
    "due_at",
    "resolved_at",
    "blocked_at",
    "unblocked_at",
    "blocks",
    "blocked_by",
    "depends_on"
]

print("GC1 JIRA FIELD CHECK")
print("=" * 60)

for field in required_fields:

    exists = field in test_ticket

    print(
        f"{field:25}",
        "OK" if exists else "MISSING"
    )

GC1 JIRA FIELD CHECK
project_id                OK
issue_id                  OK
issue_type                OK
summary                   OK
description               OK
related_requirement       OK
epic_feature              OK
status                    OK
status_history            OK
priority                  OK
assignee                  OK
team                      OK
created_at                OK
due_at                    OK
resolved_at               OK
blocked_at                OK
unblocked_at              OK
blocks                    OK
blocked_by                OK
depends_on                OK


In [ ]:
import os
import shutil

incomplete_folder = os.path.join(
    EXPORT_ROOT,
    "01_JBEAP"
)

if os.path.exists(incomplete_folder):
    shutil.rmtree(incomplete_folder)
    print("Incomplete JBEAP folder removed.")
else:
    print("No incomplete folder found.")

Incomplete JBEAP folder removed.


In [ ]:
from datetime import datetime, date
from bson import ObjectId
import numpy as np
import pandas as pd

def json_safe(obj):

    if isinstance(obj, ObjectId):
        return str(obj)

    if isinstance(obj, (datetime, date, pd.Timestamp)):
        return obj.isoformat()

    if isinstance(obj, np.bool_):
        return bool(obj)

    if isinstance(obj, np.integer):
        return int(obj)

    if isinstance(obj, np.floating):
        if np.isnan(obj):
            return None
        return float(obj)

    if isinstance(obj, float) and np.isnan(obj):
        return None

    if isinstance(obj, dict):
        return {
            str(k): json_safe(v)
            for k, v in obj.items()
        }

    if isinstance(obj, (list, tuple, set)):
        return [json_safe(v) for v in obj]

    return obj

In [ ]:
test_row = selected30.iloc[0]

test_profile = json_safe(
    test_row.to_dict()
)

print(
    json.dumps(
        test_profile,
        indent=2,
        ensure_ascii=False
    )[:5000]
)

print("\nJSON serialization test PASSED.")

{
  "repository": "RedHat",
  "project_id": "12313422",
  "project_key": "JBEAP",
  "project_name": "JBoss Enterprise Application Platform",
  "issue_count": 17102,
  "bug_count": 12612,
  "improvement_count": 0,
  "task_count": 958,
  "resolved_count": 16068,
  "unresolved_count": 1034,
  "high_priority_count": 15182,
  "issues_with_description": 15346,
  "comment_count": 50106,
  "issues_with_comments": 15533,
  "issue_link_count": 31182,
  "issues_with_links": 12672,
  "blocking_link_count": 3486,
  "history_event_count": 309227,
  "field_change_count": 460846,
  "status_change_count": 97240,
  "priority_change_count": 2321,
  "assignee_change_count": 15294,
  "resolution_change_count": 24115,
  "version_change_count": 34918,
  "reopen_count": 1835,
  "earliest_created": "2013-01-22T10:29:16",
  "latest_activity": "2022-01-04T18:14:37",
  "resolution_days_sum": 2131533.114606477,
  "resolved_with_duration": 16072,
  "issue_type_diversity": 30,
  "status_diversity": 9,
  "priority_di

In [ ]:
import os
import shutil

OLD_EXPORT_ROOT = "/content/drive/MyDrive/GC1/Selected_30_Jira_Projects"

if os.path.exists(OLD_EXPORT_ROOT):
    shutil.rmtree(OLD_EXPORT_ROOT)
    print("Old incomplete full export deleted.")
else:
    print("Old export folder not found.")

Old incomplete full export deleted.


In [ ]:
SAMPLE_ROOT = "/content/drive/MyDrive/GC1/GC1_Working_Jira_Dataset_300"

os.makedirs(SAMPLE_ROOT, exist_ok=True)

print("New working dataset folder:")
print(SAMPLE_ROOT)

New working dataset folder:
/content/drive/MyDrive/GC1/GC1_Working_Jira_Dataset_300


In [ ]:
def ticket_selection_features(doc):

    fields = doc.get("fields", {}) or {}

    # -------------------------
    # Description
    # -------------------------

    has_description = bool(
        fields.get("description")
    )

    # -------------------------
    # Comments
    # -------------------------

    comments = fields.get("comments", [])

    if not isinstance(comments, list):
        comments = []

    comment_count = len(comments)

    # -------------------------
    # Links
    # -------------------------

    links = fields.get("issuelinks", [])

    if not isinstance(links, list):
        links = []

    link_count = len(links)
    blocking_link_count = 0

    for link in links:

        if not isinstance(link, dict):
            continue

        link_type = link.get("type", {}) or {}

        relation_text = " ".join([
            str(link_type.get("name", "")),
            str(link_type.get("inward", "")),
            str(link_type.get("outward", ""))
        ]).lower()

        if (
            "block" in relation_text
            or "depend" in relation_text
        ):
            blocking_link_count += 1

    # -------------------------
    # Changelog
    # -------------------------

    changelog = doc.get("changelog", {}) or {}

    histories = changelog.get(
        "histories",
        []
    )

    if not isinstance(histories, list):
        histories = []

    history_count = len(histories)

    status_change_count = 0
    priority_change_count = 0
    assignee_change_count = 0
    reopen_count = 0

    for history in histories:

        if not isinstance(history, dict):
            continue

        for item in history.get("items", []):

            if not isinstance(item, dict):
                continue

            field = str(
                item.get("field", "")
            ).lower()

            old_value = str(
                item.get("fromString") or ""
            ).lower()

            new_value = str(
                item.get("toString") or ""
            ).lower()

            if field == "status":

                status_change_count += 1

                if (
                    new_value in ["open", "reopened"]
                    and old_value in [
                        "resolved",
                        "closed",
                        "done"
                    ]
                ):
                    reopen_count += 1

            elif field == "priority":
                priority_change_count += 1

            elif field == "assignee":
                assignee_change_count += 1

    # -------------------------
    # Priority
    # -------------------------

    priority = fields.get("priority") or {}

    priority_name = str(
        priority.get("name", "")
        if isinstance(priority, dict)
        else ""
    ).lower()

    high_priority = priority_name in [
        "blocker",
        "critical",
        "highest",
        "high",
        "major"
    ]

    # -------------------------
    # Resolution
    # -------------------------

    resolved = bool(
        fields.get("resolution")
    )

    # -------------------------
    # Selection score
    # -------------------------

    score = 0

    if has_description:
        score += 2

    score += min(comment_count, 5)

    score += min(link_count * 2, 8)

    score += min(
        blocking_link_count * 3,
        9
    )

    score += min(history_count, 10)

    score += min(
        status_change_count * 2,
        8
    )

    score += min(
        priority_change_count,
        3
    )

    score += min(
        assignee_change_count,
        3
    )

    score += min(
        reopen_count * 4,
        8
    )

    if high_priority:
        score += 3

    return {

        "score": score,

        "has_description":
            has_description,

        "comment_count":
            comment_count,

        "link_count":
            link_count,

        "blocking_link_count":
            blocking_link_count,

        "history_count":
            history_count,

        "status_change_count":
            status_change_count,

        "priority_change_count":
            priority_change_count,

        "assignee_change_count":
            assignee_change_count,

        "reopen_count":
            reopen_count,

        "high_priority":
            high_priority,

        "resolved":
            resolved,

        "created_at":
            fields.get("created")
    }

In [ ]:
TEST_REPOSITORY = "RedHat"
TEST_PROJECT = "JBEAP"

collection = db[TEST_REPOSITORY]

candidate_rows = []

cursor = collection.find({
    "fields.project.key": TEST_PROJECT
})

for doc in cursor:

    features = ticket_selection_features(
        doc
    )

    fields = doc.get(
        "fields",
        {}
    ) or {}

    candidate_rows.append({

        "issue_id":
            str(doc.get("id")),

        "issue_key":
            doc.get("key"),

        "issue_type":
            (fields.get("issuetype") or {})
            .get("name"),

        "status":
            (fields.get("status") or {})
            .get("name"),

        "priority":
            (fields.get("priority") or {})
            .get("name"),

        "summary":
            fields.get("summary"),

        **features
    })


candidate_df = pd.DataFrame(
    candidate_rows
)

print(
    "Total JBEAP tickets:",
    len(candidate_df)
)

Total JBEAP tickets: 17102


In [ ]:
print("JBEAP TICKET CHARACTERISTICS")
print("=" * 55)

print(
    "Total:",
    len(candidate_df)
)

print(
    "With descriptions:",
    int(candidate_df["has_description"].sum())
)

print(
    "With comments:",
    int(
        (candidate_df["comment_count"] > 0)
        .sum()
    )
)

print(
    "With issue links:",
    int(
        (candidate_df["link_count"] > 0)
        .sum()
    )
)

print(
    "With blocker/dependency links:",
    int(
        (
            candidate_df[
                "blocking_link_count"
            ] > 0
        ).sum()
    )
)

print(
    "With status changes:",
    int(
        (
            candidate_df[
                "status_change_count"
            ] > 0
        ).sum()
    )
)

print(
    "Reopened:",
    int(
        (
            candidate_df[
                "reopen_count"
            ] > 0
        ).sum()
    )
)

print(
    "High priority:",
    int(
        candidate_df[
            "high_priority"
        ].sum()
    )
)

print(
    "Resolved:",
    int(
        candidate_df[
            "resolved"
        ].sum()
    )
)

print(
    "Unresolved:",
    int(
        (
            ~candidate_df[
                "resolved"
            ]
        ).sum()
    )
)

JBEAP TICKET CHARACTERISTICS
Total: 17102
With descriptions: 15346
With comments: 15533
With issue links: 12672
With blocker/dependency links: 2105
With status changes: 16485
Reopened: 1537
High priority: 15182
Resolved: 16068
Unresolved: 1034


In [ ]:
display(
    candidate_df[
        "issue_type"
    ]
    .value_counts()
    .head(20)
)

,count
issue_type,
Bug,12612
Component Upgrade,1647
Enhancement,979
Task,784
Documentation,228
Feature Request,182
Requirement,171
Release,167
Sub-task,159


In [ ]:
candidate_df["created_dt"] = pd.to_datetime(
    candidate_df["created_at"],
    errors="coerce",
    utc=True
)

print(
    "Earliest ticket:",
    candidate_df["created_dt"].min()
)

print(
    "Latest ticket:",
    candidate_df["created_dt"].max()
)

print(
    "Project duration:",
    candidate_df["created_dt"].max()
    -
    candidate_df["created_dt"].min()
)

Earliest ticket: 2013-01-22 10:29:16+00:00
Latest ticket: 2022-01-03 16:31:18+00:00
Project duration: 3268 days 06:02:02


In [ ]:
import pandas as pd
import numpy as np


def select_300_tickets(candidate_df, target=300):

    df = candidate_df.copy()

    # -----------------------------------------
    # Clean dates
    # -----------------------------------------

    df["created_dt"] = pd.to_datetime(
        df["created_at"],
        errors="coerce",
        utc=True
    )

    # -----------------------------------------
    # Temporal bins
    # Divide project lifetime into 10 periods
    # -----------------------------------------

    valid_dates = df["created_dt"].dropna()

    if len(valid_dates) > 0:

        try:
            df["time_bin"] = pd.qcut(
                df["created_dt"],
                q=10,
                labels=False,
                duplicates="drop"
            )

        except:
            df["time_bin"] = 0

    else:
        df["time_bin"] = 0

    # -----------------------------------------
    # Behaviour flags
    # -----------------------------------------

    df["has_comments"] = (
        df["comment_count"] > 0
    )

    df["has_links"] = (
        df["link_count"] > 0
    )

    df["has_blocking"] = (
        df["blocking_link_count"] > 0
    )

    df["has_changes"] = (
        df["status_change_count"] > 0
    )

    df["was_reopened"] = (
        df["reopen_count"] > 0
    )

    # -----------------------------------------
    # Selection storage
    # -----------------------------------------

    selected = set()

    def add_best(subset, n):

        if n <= 0 or len(subset) == 0:
            return

        subset = subset[
            ~subset.index.isin(selected)
        ]

        subset = subset.sort_values(
            "score",
            ascending=False
        )

        for idx in subset.head(n).index:
            selected.add(idx)

    # -----------------------------------------
    # 1. BLOCKER / DEPENDENCY TICKETS
    # -----------------------------------------

    add_best(
        df[df["has_blocking"]],
        50
    )

    # -----------------------------------------
    # 2. REOPENED / REGRESSION TICKETS
    # -----------------------------------------

    add_best(
        df[df["was_reopened"]],
        40
    )

    # -----------------------------------------
    # 3. COMMENT-RICH TICKETS
    # -----------------------------------------

    comment_rich = df[
        df["comment_count"] > 0
    ].sort_values(
        "comment_count",
        ascending=False
    )

    add_best(
        comment_rich,
        40
    )

    # -----------------------------------------
    # 4. HIGH-CHANGE TICKETS
    # -----------------------------------------

    high_change = df[
        df["status_change_count"] > 0
    ].sort_values(
        [
            "status_change_count",
            "history_count"
        ],
        ascending=False
    )

    add_best(
        high_change,
        40
    )

    # -----------------------------------------
    # 5. UNRESOLVED / ACTIVE TICKETS
    # -----------------------------------------

    add_best(
        df[df["resolved"] == False],
        30
    )

    # -----------------------------------------
    # 6. RESOLVED / RECOVERY TICKETS
    # -----------------------------------------

    add_best(
        df[df["resolved"] == True],
        30
    )

    # -----------------------------------------
    # 7. ISSUE-TYPE DIVERSITY
    # -----------------------------------------

    issue_types = (
        df["issue_type"]
        .dropna()
        .unique()
    )

    for issue_type in issue_types:

        if len(selected) >= target:
            break

        type_subset = df[
            df["issue_type"] == issue_type
        ]

        # Up to 5 from each issue type
        add_best(
            type_subset,
            min(5, target - len(selected))
        )

    # -----------------------------------------
    # 8. TEMPORAL COVERAGE
    # -----------------------------------------

    time_bins = (
        df["time_bin"]
        .dropna()
        .unique()
    )

    for time_bin in sorted(time_bins):

        if len(selected) >= target:
            break

        period = df[
            df["time_bin"] == time_bin
        ]

        add_best(
            period,
            min(5, target - len(selected))
        )

    # -----------------------------------------
    # 9. FILL REMAINING SLOTS
    # -----------------------------------------

    if len(selected) < target:

        remaining = df[
            ~df.index.isin(selected)
        ].sort_values(
            "score",
            ascending=False
        )

        for idx in remaining.index:

            if len(selected) >= target:
                break

            selected.add(idx)

    # -----------------------------------------
    # Final selected sample
    # -----------------------------------------

    result = df.loc[
        list(selected)
    ].copy()

    # Hard cap
    if len(result) > target:
        result = result.sort_values(
            "score",
            ascending=False
        ).head(target)

    return result

In [ ]:
jbeap_300 = select_300_tickets(
    candidate_df,
    target=300
)

print(
    "Selected tickets:",
    len(jbeap_300)
)

print("\nSELECTION CHARACTERISTICS")
print("=" * 55)

print(
    "With descriptions:",
    int(jbeap_300["has_description"].sum())
)

print(
    "With comments:",
    int((jbeap_300["comment_count"] > 0).sum())
)

print(
    "With links:",
    int((jbeap_300["link_count"] > 0).sum())
)

print(
    "With blocker/dependency links:",
    int(
        (
            jbeap_300[
                "blocking_link_count"
            ] > 0
        ).sum()
    )
)

print(
    "With status changes:",
    int(
        (
            jbeap_300[
                "status_change_count"
            ] > 0
        ).sum()
    )
)

print(
    "Reopened:",
    int(
        (
            jbeap_300[
                "reopen_count"
            ] > 0
        ).sum()
    )
)

print(
    "High priority:",
    int(
        jbeap_300[
            "high_priority"
        ].sum()
    )
)

print(
    "Resolved:",
    int(
        jbeap_300[
            "resolved"
        ].sum()
    )
)

print(
    "Unresolved:",
    int(
        (~jbeap_300["resolved"]).sum()
    )
)

Selected tickets: 300

SELECTION CHARACTERISTICS
With descriptions: 278
With comments: 287
With links: 289
With blocker/dependency links: 225
With status changes: 297
Reopened: 141
High priority: 288
Resolved: 258
Unresolved: 42


In [ ]:
display(
    jbeap_300[
        "issue_type"
    ].value_counts()
)

,count
issue_type,
Bug,179
Component Upgrade,21
Enhancement,17
Task,14
Requirement,11
Feature Request,8
Epic,7
Tracker,7
QE Sub-task,5


In [ ]:
print("Original project:")
print(
    candidate_df["created_dt"].min(),
    "→",
    candidate_df["created_dt"].max()
)

print("\nSelected 300:")
print(
    jbeap_300["created_dt"].min(),
    "→",
    jbeap_300["created_dt"].max()
)

print("\nTickets per temporal bin:")

print(
    jbeap_300["time_bin"]
    .value_counts()
    .sort_index()
)

Original project:
2013-01-22 10:29:16+00:00 → 2022-01-03 16:31:18+00:00

Selected 300:
2013-07-22 08:29:45+00:00 → 2021-11-24 13:49:40+00:00

Tickets per temporal bin:
time_bin
0    33
1    60
2    31
3    44
4    35
5    27
6    16
7    17
8    15
9    22
Name: count, dtype: int64


In [ ]:
import os
import json
import pandas as pd
import numpy as np

WORKING_ROOT = "/content/drive/MyDrive/GC1/GC1_Working_Jira_Dataset_300"

os.makedirs(WORKING_ROOT, exist_ok=True)

print("Working dataset folder:")
print(WORKING_ROOT)

print("Projects to process:", len(selected30))

Working dataset folder:
/content/drive/MyDrive/GC1/GC1_Working_Jira_Dataset_300
Projects to process: 30


In [ ]:
def build_candidate_table(repository, project_key):

    collection = db[repository]

    rows = []

    cursor = collection.find({
        "fields.project.key": project_key
    })

    for doc in cursor:

        fields = doc.get("fields", {}) or {}

        features = ticket_selection_features(doc)

        issue_type = fields.get("issuetype") or {}
        status = fields.get("status") or {}
        priority = fields.get("priority") or {}

        rows.append({

            "issue_id": str(doc.get("id")),
            "issue_key": doc.get("key"),

            "issue_type":
                issue_type.get("name")
                if isinstance(issue_type, dict)
                else None,

            "status":
                status.get("name")
                if isinstance(status, dict)
                else None,

            "priority":
                priority.get("name")
                if isinstance(priority, dict)
                else None,

            "summary": fields.get("summary"),

            **features
        })

    candidate_df = pd.DataFrame(rows)

    if len(candidate_df) == 0:
        return candidate_df

    candidate_df["created_dt"] = pd.to_datetime(
        candidate_df["created_at"],
        errors="coerce",
        utc=True
    )

    return candidate_df

In [ ]:
def select_300_tickets(candidate_df, target=300):

    df = candidate_df.copy()

    if len(df) <= target:
        df["time_bin"] = 0
        return df

    # =====================================================
    # TEMPORAL BINS
    # =====================================================

    df["created_dt"] = pd.to_datetime(
        df["created_at"],
        errors="coerce",
        utc=True
    )

    try:
        df["time_bin"] = pd.qcut(
            df["created_dt"],
            q=10,
            labels=False,
            duplicates="drop"
        )
    except:
        df["time_bin"] = 0

    # =====================================================
    # BEHAVIOURAL FLAGS
    # =====================================================

    df["has_comments"] = (
        df["comment_count"] > 0
    )

    df["has_links"] = (
        df["link_count"] > 0
    )

    df["has_blocking"] = (
        df["blocking_link_count"] > 0
    )

    df["has_changes"] = (
        df["status_change_count"] > 0
    )

    df["was_reopened"] = (
        df["reopen_count"] > 0
    )

    selected = set()

    # =====================================================
    # HELPER
    # =====================================================

    def add_best(subset, n, sort_columns=None):

        if n <= 0 or len(subset) == 0:
            return

        subset = subset[
            ~subset.index.isin(selected)
        ].copy()

        if len(subset) == 0:
            return

        if sort_columns:

            subset = subset.sort_values(
                sort_columns,
                ascending=[False] * len(sort_columns)
            )

        else:

            subset = subset.sort_values(
                "score",
                ascending=False
            )

        for idx in subset.head(n).index:

            if len(selected) >= target:
                break

            selected.add(idx)

    # =====================================================
    # 1. FORCE EARLY PROJECT COVERAGE
    # =====================================================

    earliest = df.sort_values(
        "created_dt",
        ascending=True
    )

    add_best(
        earliest.head(100),
        10
    )

    # =====================================================
    # 2. FORCE LATE PROJECT COVERAGE
    # =====================================================

    latest = df.sort_values(
        "created_dt",
        ascending=False
    )

    add_best(
        latest.head(100),
        10
    )

    # =====================================================
    # 3. DEPENDENCY / BLOCKER
    # =====================================================

    add_best(
        df[df["has_blocking"]],
        50,
        [
            "blocking_link_count",
            "link_count",
            "score"
        ]
    )

    # =====================================================
    # 4. REOPEN / REGRESSION
    # =====================================================

    add_best(
        df[df["was_reopened"]],
        40,
        [
            "reopen_count",
            "status_change_count",
            "score"
        ]
    )

    # =====================================================
    # 5. COMMENT-RICH
    # =====================================================

    add_best(
        df[df["has_comments"]],
        40,
        [
            "comment_count",
            "score"
        ]
    )

    # =====================================================
    # 6. HIGH CHANGE
    # =====================================================

    add_best(
        df[df["has_changes"]],
        40,
        [
            "status_change_count",
            "history_count",
            "score"
        ]
    )

    # =====================================================
    # 7. UNRESOLVED
    # =====================================================

    add_best(
        df[df["resolved"] == False],
        30
    )

    # =====================================================
    # 8. RESOLVED
    # =====================================================

    add_best(
        df[df["resolved"] == True],
        25
    )

    # =====================================================
    # 9. ISSUE-TYPE DIVERSITY
    # =====================================================

    issue_types = (
        df["issue_type"]
        .dropna()
        .value_counts()
        .index
        .tolist()
    )

    for issue_type in issue_types:

        if len(selected) >= target:
            break

        subset = df[
            df["issue_type"] == issue_type
        ]

        add_best(
            subset,
            min(
                5,
                target - len(selected)
            )
        )

    # =====================================================
    # 10. TEMPORAL COVERAGE
    # =====================================================

    available_bins = sorted(
        df["time_bin"]
        .dropna()
        .unique()
    )

    for time_bin in available_bins:

        if len(selected) >= target:
            break

        subset = df[
            df["time_bin"] == time_bin
        ]

        add_best(
            subset,
            min(
                5,
                target - len(selected)
            )
        )

    # =====================================================
    # 11. FILL REMAINING WITH BEST TICKETS
    # =====================================================

    if len(selected) < target:

        remaining = df[
            ~df.index.isin(selected)
        ].sort_values(
            "score",
            ascending=False
        )

        for idx in remaining.index:

            if len(selected) >= target:
                break

            selected.add(idx)

    result = df.loc[
        list(selected)
    ].copy()

    # Safety
    if len(result) > target:

        result = result.sort_values(
            "score",
            ascending=False
        ).head(target)

    return result

In [ ]:
def selection_reason(row):

    reasons = []

    if row.get("blocking_link_count", 0) > 0:
        reasons.append("dependency/blocker")

    if row.get("reopen_count", 0) > 0:
        reasons.append("reopened/regression")

    if row.get("comment_count", 0) > 0:
        reasons.append("comment-rich")

    if row.get("status_change_count", 0) > 0:
        reasons.append("status-change")

    if row.get("high_priority", False):
        reasons.append("high-priority")

    if row.get("resolved", False):
        reasons.append("resolved")
    else:
        reasons.append("unresolved")

    if row.get("has_description", False):
        reasons.append("description")

    return ", ".join(reasons)

In [ ]:
test_candidates = build_candidate_table(
    "RedHat",
    "JBEAP"
)

test_sample = select_300_tickets(
    test_candidates,
    target=300
)

test_sample["selection_reason"] = (
    test_sample.apply(
        selection_reason,
        axis=1
    )
)

print("Candidates:", len(test_candidates))
print("Selected:", len(test_sample))

display(
    test_sample[
        [
            "issue_key",
            "issue_type",
            "status",
            "priority",
            "score",
            "selection_reason"
        ]
    ].head(20)
)

Candidates: 17102
Selected: 300


,issue_key,issue_type,status,priority,score,selection_reason
14848,JBEAP-2860,Bug,Closed,Blocker,36,"reopened/regression, comment-rich, status-chan..."
14850,JBEAP-2858,Task,Closed,Blocker,47,"dependency/blocker, comment-rich, status-chang..."
8195,JBEAP-10985,Bug,Closed,Blocker,47,"dependency/blocker, comment-rich, status-chang..."
14338,JBEAP-3432,Bug,Verified,Major,38,"comment-rich, status-change, high-priority, re..."
11267,JBEAP-7174,Requirement,Closed,Blocker,32,"comment-rich, status-change, high-priority, re..."
11268,JBEAP-7173,Requirement,Closed,Major,29,"comment-rich, status-change, high-priority, re..."
15879,JBEAP-1667,Enhancement,Closed,Critical,40,"reopened/regression, comment-rich, status-chan..."
11271,JBEAP-7170,Requirement,Closed,Blocker,32,"comment-rich, status-change, high-priority, re..."
14857,JBEAP-2847,Bug,Closed,Critical,36,"comment-rich, status-change, high-priority, re..."
11272,JBEAP-7169,Requirement,Closed,Blocker,48,"dependency/blocker, reopened/regression, comme..."


In [ ]:
import os
import json

sampling_summary = []

print("=" * 75)
print("STARTING GC1 300-TICKET DATASET EXPORT")
print("=" * 75)


for project_number, (_, project_row) in enumerate(
    selected30.iterrows(),
    start=1
):

    repository = project_row["repository"]
    project_key = project_row["project_key"]
    project_name = project_row["project_name"]

    print()
    print("=" * 75)
    print(
        f"[{project_number:02d}/30] "
        f"{project_name} ({project_key})"
    )
    print("Repository:", repository)
    print("=" * 75)

    # =====================================================
    # A. BUILD CANDIDATE TABLE
    # =====================================================

    candidate_df_project = (
        build_candidate_table(
            repository,
            project_key
        )
    )

    total_available = len(
        candidate_df_project
    )

    print(
        "Available tickets:",
        f"{total_available:,}"
    )

    # =====================================================
    # B. SELECT ~300
    # =====================================================

    selected_tickets = (
        select_300_tickets(
            candidate_df_project,
            target=300
        )
    )

    selected_tickets[
        "selection_reason"
    ] = selected_tickets.apply(
        selection_reason,
        axis=1
    )

    selected_keys = set(
        selected_tickets[
            "issue_key"
        ].dropna()
    )

    print(
        "Selected tickets:",
        len(selected_keys)
    )

    # =====================================================
    # C. CREATE PROJECT FOLDER
    # =====================================================

    safe_key = str(
        project_key
    ).replace("/", "_")

    project_folder = os.path.join(
        WORKING_ROOT,
        f"{project_number:02d}_{safe_key}"
    )

    os.makedirs(
        project_folder,
        exist_ok=True
    )

    raw_path = os.path.join(
        project_folder,
        "raw_issues.jsonl"
    )

    standardized_path = os.path.join(
        project_folder,
        "jira_tickets.jsonl"
    )

    manifest_path = os.path.join(
        project_folder,
        "selected_ticket_manifest.csv"
    )

    profile_path = os.path.join(
        project_folder,
        "sampling_profile.json"
    )

    # =====================================================
    # D. SAVE TICKET MANIFEST
    # =====================================================

    manifest_columns = [
        "issue_id",
        "issue_key",
        "issue_type",
        "status",
        "priority",
        "summary",
        "score",
        "has_description",
        "comment_count",
        "link_count",
        "blocking_link_count",
        "history_count",
        "status_change_count",
        "priority_change_count",
        "assignee_change_count",
        "reopen_count",
        "high_priority",
        "resolved",
        "created_at",
        "time_bin",
        "selection_reason"
    ]

    available_columns = [
        c for c in manifest_columns
        if c in selected_tickets.columns
    ]

    selected_tickets[
        available_columns
    ].to_csv(
        manifest_path,
        index=False
    )

    # =====================================================
    # E. RETRIEVE COMPLETE ORIGINAL DOCUMENTS
    # =====================================================

    collection = db[repository]

    cursor = collection.find({
        "fields.project.key":
            project_key,

        "key": {
            "$in": list(selected_keys)
        }
    })

    exported = 0

    with open(
        raw_path,
        "w",
        encoding="utf-8"
    ) as raw_file, open(
        standardized_path,
        "w",
        encoding="utf-8"
    ) as standardized_file:

        for doc in cursor:

            # ---------------------------------------------
            # RAW ORIGINAL RECORD
            # ---------------------------------------------

            raw_file.write(
                json.dumps(
                    json_safe(doc),
                    ensure_ascii=False
                )
                + "\n"
            )

            # ---------------------------------------------
            # GC1 STANDARDIZED RECORD
            # ---------------------------------------------

            ticket = make_gc1_ticket(
                doc,
                repository
            )

            standardized_file.write(
                json.dumps(
                    json_safe(ticket),
                    ensure_ascii=False
                )
                + "\n"
            )

            exported += 1

    # =====================================================
    # F. SAMPLING PROFILE
    # =====================================================

    sample = selected_tickets

    sampling_profile = {

        "repository":
            repository,

        "project_id":
            json_safe(
                project_row.get(
                    "project_id"
                )
            ),

        "project_key":
            project_key,

        "project_name":
            project_name,

        "total_available_tickets":
            int(total_available),

        "target_sample_size":
            300,

        "exported_ticket_count":
            int(exported),

        "tickets_with_description":
            int(
                sample[
                    "has_description"
                ].sum()
            ),

        "tickets_with_comments":
            int(
                (
                    sample[
                        "comment_count"
                    ] > 0
                ).sum()
            ),

        "tickets_with_links":
            int(
                (
                    sample[
                        "link_count"
                    ] > 0
                ).sum()
            ),

        "tickets_with_blocker_dependency":
            int(
                (
                    sample[
                        "blocking_link_count"
                    ] > 0
                ).sum()
            ),

        "tickets_with_status_changes":
            int(
                (
                    sample[
                        "status_change_count"
                    ] > 0
                ).sum()
            ),

        "reopened_tickets":
            int(
                (
                    sample[
                        "reopen_count"
                    ] > 0
                ).sum()
            ),

        "high_priority_tickets":
            int(
                sample[
                    "high_priority"
                ].sum()
            ),

        "resolved_tickets":
            int(
                sample[
                    "resolved"
                ].sum()
            ),

        "unresolved_tickets":
            int(
                (
                    ~sample[
                        "resolved"
                    ]
                ).sum()
            ),

        "sample_earliest_date":
            (
                sample["created_dt"]
                .min()
                .isoformat()
                if sample[
                    "created_dt"
                ].notna().any()
                else None
            ),

        "sample_latest_date":
            (
                sample["created_dt"]
                .max()
                .isoformat()
                if sample[
                    "created_dt"
                ].notna().any()
                else None
            ),

        "issue_type_distribution":
            {
                str(k): int(v)
                for k, v in
                sample[
                    "issue_type"
                ]
                .value_counts(
                    dropna=False
                )
                .items()
            },

        "sampling_method":
            (
                "Information-rich, behaviourally diverse "
                "and temporally distributed Jira issue sampling."
            )
    }

    with open(
        profile_path,
        "w",
        encoding="utf-8"
    ) as profile_file:

        json.dump(
            json_safe(
                sampling_profile
            ),
            profile_file,
            indent=2,
            ensure_ascii=False
        )

    # =====================================================
    # G. SUMMARY
    # =====================================================

    sampling_summary.append({

        "project_number":
            project_number,

        "repository":
            repository,

        "project_key":
            project_key,

        "project_name":
            project_name,

        "available_tickets":
            total_available,

        "selected_tickets":
            len(selected_keys),

        "exported_tickets":
            exported,

        "folder":
            project_folder
    })

    if exported == len(selected_keys):

        print(
            f"SUCCESS: "
            f"{exported} tickets exported."
        )

    else:

        print(
            f"WARNING: Selected "
            f"{len(selected_keys)}, "
            f"exported {exported}."
        )


print()
print("=" * 75)
print("ALL 30 PROJECTS FINISHED")
print("=" * 75)

STARTING GC1 300-TICKET DATASET EXPORT

[01/30] JBoss Enterprise Application Platform (JBEAP)
Repository: RedHat
Available tickets: 17,102
Selected tickets: 300
SUCCESS: 300 tickets exported.

[02/30] Tools (JBoss Tools) (JBIDE)
Repository: RedHat
Available tickets: 27,295
Selected tickets: 300
SUCCESS: 300 tickets exported.

[03/30] Sakai (SAK)
Repository: Sakai
Available tickets: 43,351
Selected tickets: 300
SUCCESS: 300 tickets exported.

[04/30] Red Hat CodeReady Studio (devstudio) (JBDS)
Repository: RedHat
Available tickets: 4,221
Selected tickets: 300
SUCCESS: 300 tickets exported.

[05/30] Maven (MNG)
Repository: Apache
Available tickets: 6,046
Selected tickets: 300
SUCCESS: 300 tickets exported.

[06/30] Spark (SPARK)
Repository: Apache
Available tickets: 37,443
Selected tickets: 300
SUCCESS: 300 tickets exported.

[07/30] Red Hat Fuse (ENTESB)
Repository: RedHat
Available tickets: 13,529
Selected tickets: 300
SUCCESS: 300 tickets exported.

[08/30] Spring Framework (SPR)
Repos

In [ ]:
sampling_manifest = pd.DataFrame(
    sampling_summary
)

OVERALL_MANIFEST = os.path.join(
    WORKING_ROOT,
    "sampling_manifest.csv"
)

sampling_manifest.to_csv(
    OVERALL_MANIFEST,
    index=False
)

display(sampling_manifest)

print(
    "\nTotal exported tickets:",
    sampling_manifest[
        "exported_tickets"
    ].sum()
)

print(
    "\nManifest saved to:"
)

print(OVERALL_MANIFEST)

,project_number,repository,project_key,project_name,available_tickets,selected_tickets,exported_tickets,folder
0,1,RedHat,JBEAP,JBoss Enterprise Application Platform,17102,300,300,/content/drive/MyDrive/GC1/GC1_Working_Jira_Da...
1,2,RedHat,JBIDE,Tools (JBoss Tools),27295,300,300,/content/drive/MyDrive/GC1/GC1_Working_Jira_Da...
2,3,Sakai,SAK,Sakai,43351,300,300,/content/drive/MyDrive/GC1/GC1_Working_Jira_Da...
3,4,RedHat,JBDS,Red Hat CodeReady Studio (devstudio),4221,300,300,/content/drive/MyDrive/GC1/GC1_Working_Jira_Da...
4,5,Apache,MNG,Maven,6046,300,300,/content/drive/MyDrive/GC1/GC1_Working_Jira_Da...
5,6,Apache,SPARK,Spark,37443,300,300,/content/drive/MyDrive/GC1/GC1_Working_Jira_Da...
6,7,RedHat,ENTESB,Red Hat Fuse,13529,300,300,/content/drive/MyDrive/GC1/GC1_Working_Jira_Da...
7,8,Spring,SPR,Spring Framework,17413,300,300,/content/drive/MyDrive/GC1/GC1_Working_Jira_Da...
8,9,Apache,AMBARI,Ambari,25384,300,300,/content/drive/MyDrive/GC1/GC1_Working_Jira_Da...
9,10,Mojang,MC,Minecraft: Java Edition,213845,300,300,/content/drive/MyDrive/GC1/GC1_Working_Jira_Da...



Total exported tickets: 9000

Manifest saved to:
/content/drive/MyDrive/GC1/GC1_Working_Jira_Dataset_300/sampling_manifest.csv


In [ ]:
sampling_manifest[
    "aligned"
] = (
    sampling_manifest[
        "selected_tickets"
    ]
    ==
    sampling_manifest[
        "exported_tickets"
    ]
)

display(
    sampling_manifest[
        [
            "project_key",
            "available_tickets",
            "selected_tickets",
            "exported_tickets",
            "aligned"
        ]
    ]
)

print(
    "\nAll projects aligned:",
    sampling_manifest[
        "aligned"
    ].all()
)

,project_key,available_tickets,selected_tickets,exported_tickets,aligned
0,JBEAP,17102,300,300,True
1,JBIDE,27295,300,300,True
2,SAK,43351,300,300,True
3,JBDS,4221,300,300,True
4,MNG,6046,300,300,True
5,SPARK,37443,300,300,True
6,ENTESB,13529,300,300,True
7,SPR,17413,300,300,True
8,AMBARI,25384,300,300,True
9,MC,213845,300,300,True



All projects aligned: True


In [ ]:
import os
import json
import pandas as pd

quality_rows = []

for _, project in sampling_manifest.iterrows():

    project_key = project["project_key"]
    folder = project["folder"]

    jira_file = os.path.join(
        folder,
        "jira_tickets.jsonl"
    )

    with open(
        jira_file,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            ticket = json.loads(line)

            quality_rows.append({

                "project_key":
                    project_key,

                "issue_key":
                    ticket.get("issue_key"),

                "has_summary":
                    bool(ticket.get("summary")),

                "has_description":
                    bool(ticket.get("description")),

                "has_issue_type":
                    bool(ticket.get("issue_type")),

                "has_status":
                    bool(ticket.get("status")),

                "has_priority":
                    bool(ticket.get("priority")),

                "has_assignee":
                    bool(ticket.get("assignee")),

                "has_created_at":
                    bool(ticket.get("created_at")),

                "has_updated_at":
                    bool(ticket.get("updated_at")),

                "has_resolved_at":
                    bool(ticket.get("resolved_at")),

                "has_status_history":
                    len(
                        ticket.get(
                            "status_history",
                            []
                        )
                    ) > 0,

                "has_comments":
                    len(
                        ticket.get(
                            "comments",
                            []
                        )
                    ) > 0,

                "has_links":
                    (
                        len(ticket.get("blocks", []))
                        + len(ticket.get("blocked_by", []))
                        + len(ticket.get("depends_on", []))
                        + len(ticket.get("other_issue_links", []))
                    ) > 0,

                "has_block_dependency":
                    (
                        len(ticket.get("blocks", []))
                        + len(ticket.get("blocked_by", []))
                        + len(ticket.get("depends_on", []))
                    ) > 0,

                "has_changelog":
                    len(
                        ticket.get(
                            "full_changelog",
                            []
                        )
                    ) > 0
            })


quality_df = pd.DataFrame(
    quality_rows
)

print("Tickets checked:", len(quality_df))

Tickets checked: 9000


In [ ]:
coverage_columns = [
    "has_summary",
    "has_description",
    "has_issue_type",
    "has_status",
    "has_priority",
    "has_assignee",
    "has_created_at",
    "has_updated_at",
    "has_resolved_at",
    "has_status_history",
    "has_comments",
    "has_links",
    "has_block_dependency",
    "has_changelog"
]

coverage = (
    quality_df[coverage_columns]
    .mean()
    .mul(100)
    .round(2)
)

print("GC1 JIRA DATASET FIELD COVERAGE")
print("=" * 55)

for field, percentage in coverage.items():

    print(
        f"{field:30} "
        f"{percentage:6.2f}%"
    )

GC1 JIRA DATASET FIELD COVERAGE
has_summary                    100.00%
has_description                 93.96%
has_issue_type                 100.00%
has_status                     100.00%
has_priority                    89.79%
has_assignee                    82.80%
has_created_at                 100.00%
has_updated_at                 100.00%
has_resolved_at                 82.83%
has_status_history              97.49%
has_comments                    95.70%
has_links                       74.23%
has_block_dependency            36.97%
has_changelog                   99.96%


In [ ]:
project_quality = (
    quality_df
    .groupby("project_key")[
        coverage_columns
    ]
    .mean()
    .mul(100)
    .round(1)
)

display(project_quality)

,has_summary,has_description,has_issue_type,has_status,has_priority,has_assignee,has_created_at,has_updated_at,has_resolved_at,has_status_history,has_comments,has_links,has_block_dependency,has_changelog
project_key,,,,,,,,,,,,,,
AEROGEAR,100.0,89.3,100.0,100.0,100.0,68.0,100.0,100.0,89.0,94.7,83.7,59.7,38.3,100.0
AMBARI,100.0,92.7,100.0,100.0,100.0,94.3,100.0,100.0,82.7,96.0,95.3,38.3,22.3,100.0
BXMSDOC,100.0,95.0,100.0,100.0,99.7,99.7,100.0,100.0,86.3,98.0,93.7,48.3,16.3,100.0
CONFSERVER,100.0,100.0,100.0,100.0,66.3,68.7,100.0,100.0,76.7,99.7,96.3,92.0,24.0,100.0
DAOS,100.0,96.3,100.0,100.0,100.0,100.0,100.0,100.0,83.3,96.0,93.7,79.7,66.0,100.0
DOCS,100.0,90.7,100.0,100.0,100.0,92.0,100.0,100.0,87.0,99.0,90.7,63.7,24.3,100.0
ENTESB,100.0,93.7,100.0,100.0,100.0,83.3,100.0,100.0,88.0,100.0,93.0,71.7,33.3,100.0
EVG,100.0,83.7,100.0,100.0,100.0,100.0,100.0,100.0,84.0,97.7,92.7,77.0,46.7,100.0
FLEX,100.0,97.0,100.0,100.0,100.0,63.7,100.0,100.0,77.7,91.7,94.7,34.7,17.0,98.7


In [ ]:
QUALITY_PATH = (
    "/content/drive/MyDrive/GC1/"
    "GC1_Working_Jira_Dataset_300/"
    "jira_data_quality_report.csv"
)

project_quality.to_csv(
    QUALITY_PATH
)

print("Quality report saved:")
print(QUALITY_PATH)

Quality report saved:
/content/drive/MyDrive/GC1/GC1_Working_Jira_Dataset_300/jira_data_quality_report.csv


In [ ]:
import os
import json

print("=" * 70)
print("CREATING FULL 300-TICKET JSON FILES")
print("=" * 70)

created_files = []

for _, row in sampling_manifest.iterrows():

    project_key = row["project_key"]
    project_name = row["project_name"]
    folder = row["folder"]

    # Existing standardized JSONL file
    jsonl_path = os.path.join(
        folder,
        "jira_tickets.jsonl"
    )

    # New readable JSON file
    output_json = os.path.join(
        folder,
        "all_300_tickets_full.json"
    )

    tickets = []

    # Read all 300 tickets
    with open(
        jsonl_path,
        "r",
        encoding="utf-8"
    ) as f:

        for line in f:

            line = line.strip()

            if line:
                tickets.append(
                    json.loads(line)
                )

    # Write as normal formatted JSON array
    with open(
        output_json,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            tickets,
            f,
            indent=2,
            ensure_ascii=False
        )

    created_files.append({
        "project_key": project_key,
        "project_name": project_name,
        "tickets": len(tickets),
        "json_file": output_json
    })

    print(
        f"{project_key:15} "
        f"-> {len(tickets):3} tickets created"
    )


print()
print("=" * 70)
print("COMPLETED")
print("=" * 70)
print("Projects processed:", len(created_files))

CREATING FULL 300-TICKET JSON FILES
JBEAP           -> 300 tickets created
JBIDE           -> 300 tickets created
SAK             -> 300 tickets created
JBDS            -> 300 tickets created
MNG             -> 300 tickets created
SPARK           -> 300 tickets created
ENTESB          -> 300 tickets created
SPR             -> 300 tickets created
AMBARI          -> 300 tickets created
MC              -> 300 tickets created
OSSRH           -> 300 tickets created
INFRA           -> 300 tickets created
FLEX            -> 300 tickets created
BXMSDOC         -> 300 tickets created
MONGOID         -> 300 tickets created
QTBUG           -> 300 tickets created
JRASERVER       -> 300 tickets created
HBASE           -> 300 tickets created
CONFSERVER      -> 300 tickets created
SERVER          -> 300 tickets created
JRACLOUD        -> 300 tickets created
AEROGEAR        -> 300 tickets created
WT              -> 300 tickets created
DOCS            -> 300 tickets created
NEXUS           -> 300 ticke

In [ ]:
verification = []

for item in created_files:

    file_path = item["json_file"]

    with open(
        file_path,
        "r",
        encoding="utf-8"
    ) as f:

        data = json.load(f)

    verification.append({
        "project_key": item["project_key"],
        "ticket_count": len(data),
        "correct": len(data) == 300
    })


import pandas as pd

json_verification = pd.DataFrame(
    verification
)

display(json_verification)

print(
    "\nAll 30 projects contain 300 tickets:",
    json_verification["correct"].all()
)

,project_key,ticket_count,correct
0,JBEAP,300,True
1,JBIDE,300,True
2,SAK,300,True
3,JBDS,300,True
4,MNG,300,True
5,SPARK,300,True
6,ENTESB,300,True
7,SPR,300,True
8,AMBARI,300,True
9,MC,300,True



All 30 projects contain 300 tickets: True
